# POF 2017-2018 — Escolaridade e Endividamento Familiar

**TCC — Universidade de São Paulo**

Este notebook lê os arquivos brutos da POF, constrói as variáveis analíticas e roda os modelos econométricos.

### Estrutura
1. Configuração de caminhos e mapeamento de arquivos
2. Funções de leitura dos arquivos TXT em formato posicional
3. Carregamento de todos os arquivos POF
4. Definição dos códigos de dívida
5. Agregação por UC e construção das variáveis analíticas
6. Validação das dívidas agregadas
7. Análise descritiva
8. Modelos de regressão (abordagem em duas partes + OLS log-linear)
9. Diagnósticos do modelo


## 1. Configuração

In [1]:
import pandas as pd
import polars as pl
import os
from pathlib import Path

# Use relative paths: assume notebook is run from the repository root
base_path = Path('.').resolve()
pasta_txt = base_path / "Dados_20230713"
caminho_excel = base_path / "Dicionários de váriaveis.xlsx"

# Mapeamento dos arquivos TXT para os seus respectivos Dicionários (abas do Excel)
arquivos_pof = {
    # "DOMICILIO.TXT": "Domicílio",
    "MORADOR.TXT": "Morador",
    # "MORADOR_QUALI_VIDA.TXT": "Morador - Qualidade de Vida",
    # "ALUGUEL_ESTIMADO.TXT": "Aluguel Estimado",
    "DESPESA_COLETIVA.TXT": "Despesa Coletiva",
    # "CADERNETA_COLETIVA.TXT": "Caderneta Coletiva",
    "DESPESA_INDIVIDUAL.TXT": "Despesa Individual",
    # "RENDIMENTO_TRABALHO.TXT": "Rendimento do Trabalho",
    # "OUTROS_RENDIMENTOS.TXT": "Outros Rendimentos",
    # "CONSUMO_ALIMENTAR.TXT": "Consumo Alimentar",
    # "SERVICO_NAO_MONETARIO_POF2.TXT": "Serviços Não Monetários - POF 2",
    # "SERVICO_NAO_MONETARIO_POF4.TXT": "Serviços Não Monetários - POF 4",
    # "CARACTERISTICAS_DIETA.TXT": "Características da Dieta",
    # "CONDICOES_VIDA.TXT": "Condições de Vida",
    # "INVENTARIO.TXT": "Inventário",
    # "RESTRICAO_PRODUTOS_SERVICOS_SAUDE.TXT": "Restrição - Saúde"
}

## 2. Funções de leitura

In [2]:
def extrair_layout_do_dicionario_excel(caminho_excel, aba):
    """
    Lê a aba do dicionário Excel e retorna uma lista de tuplas
    (nome_variável, posição_inicial_0indexed, tamanho, divisor_decimal).
    """
    df_dic = pd.read_excel(caminho_excel, sheet_name=aba, dtype=str)

    # Localiza a linha que contém o cabeçalho real da tabela
    try:
        header_idx = df_dic[df_dic.apply(
            lambda r: r.astype(str).str.contains('Posição Inicial', na=False).any(),
            axis=1
        )].index[0]

        df_dic.columns = df_dic.iloc[header_idx]
        df_dic = df_dic.iloc[header_idx + 1:].copy()
    except IndexError:
        print(f"⚠️ Erro ao encontrar cabeçalho na aba: {aba}")
        return []

    df_dic = df_dic.dropna(subset=['Posição Inicial', 'Código da variável'])
    df_dic = df_dic[df_dic['Posição Inicial'].astype(str).str.isnumeric()]

    layout = []
    for _, row in df_dic.iterrows():
        nome = str(row['Código da variável']).strip()
        pos = int(row['Posição Inicial']) - 1  # converte para índice 0-based
        tamanho = int(row['Tamanho'])

        decimais = row.get('Decimais', None)
        div = 1
        if pd.notna(decimais) and str(decimais).strip().isnumeric():
            div = 10 ** int(float(str(decimais).strip()))

        layout.append((nome, pos, tamanho, div))

    return layout


def ler_pof_txt_com_polars(caminho_txt, layout):
    """
    Lê arquivo TXT de largura fixa da POF usando Polars.
    Cada linha é lida como uma única string 'raw', depois fatiada
    conforme o layout extraído do dicionário.
    """
    print(f"Lendo o arquivo: {caminho_txt}...")

    df = pl.read_csv(
        caminho_txt,
        has_header=False,
        new_columns=["raw"],
        truncate_ragged_lines=True,
        quote_char=None
    )

    expressoes = []
    for nome, start, length, div_decimal in layout:
        col_expr = pl.col("raw").str.slice(start, length).str.strip_chars()
        col_expr = pl.when(col_expr == "").then(None).otherwise(col_expr)

        if div_decimal > 1:
            # Variável numérica com casas decimais implícitas (ex.: valores monetários)
            col_expr = (col_expr.cast(pl.Float64, strict=False) / div_decimal).alias(nome)
        else:
            col_expr = col_expr.alias(nome)

        expressoes.append(col_expr)

    df_estruturado = df.select(expressoes)
    return df_estruturado

## 3. Carregamento de todos os arquivos POF

In [3]:
dataframes_pof = {}

# Diretório onde armazenamos Parquet gerados
dados_parquet_dir = base_path / "DadosParquet"
dados_parquet_dir.mkdir(parents=True, exist_ok=True)

for arquivo_txt, aba_excel in arquivos_pof.items():

    nome_chave = arquivo_txt.split(".")[0]
    parquet_path = dados_parquet_dir / f"{nome_chave}.parquet"
    caminho_txt = pasta_txt / arquivo_txt

    # Se já existe Parquet gerado, lê direto e pula a leitura do TXT
    if parquet_path.exists():
        df_polars = pl.read_parquet(parquet_path)
        dataframes_pof[nome_chave] = df_polars
        print(f"[{nome_chave}] lido parquet -> {df_polars.shape}")
        continue

    # Caso não haja Parquet, tenta ler o TXT e gerar o Parquet
    if caminho_txt.exists():
        layout_atual = extrair_layout_do_dicionario_excel(caminho_excel, aba_excel)

        if not layout_atual:
            continue

        df_polars = ler_pof_txt_com_polars(str(caminho_txt), layout_atual)

        dataframes_pof[nome_chave] = df_polars
        print(f"[{nome_chave}] OK -> {df_polars.shape}")

        # grava Parquet para próximas execuções
        df_polars.write_parquet(str(parquet_path))

    else:
        print(f"⚠️ TXT não encontrado: {caminho_txt}")

[MORADOR] lido parquet -> (178431, 56)
[DESPESA_COLETIVA] lido parquet -> (478572, 27)
[DESPESA_INDIVIDUAL] lido parquet -> (1836032, 25)


## 4. Definição dos códigos de dívida

Os códigos abaixo identificam as despesas que representam encargos financeiros
(juros, tarifas bancárias, pagamentos de empréstimos, etc.) dentro da POF.
São usados para filtrar `DESPESA_INDIVIDUAL` e `DESPESA_COLETIVA`.

# checar pra cada divida de custo sozinha as barras la da relacao de estudo e quanto eh gasto pra ver quais tao maiores na galera rica

In [6]:
dividas_custo = {
    "2600101": "JUROS DE CHEQUE ESPECIAL",
    "2600201": "JUROS DE CARTAO DE CREDITO",
    # "2600401": "SEGURO DE CARTAO DE CREDITO",
    # "2600503": "MANUTENCAO DE CHEQUE ESPECIAL",
    # "2600801": "TAXA DE CARTAO ESPECIAL",
    # "2601103": "RENOVACAO DE CHEQUE ESPECIAL",
    # "2601201": "TAXA DE DEVOLUCAO DE CHEQUE",
    "4800201": "JUROS DE EMPRESTIMO",
    # "4800301": "SEGURO DE EMPRESTIMO",
    "5506001": "JUROS DE EMPRESTIMO"
}

dividas_amortizacao = {
    # "4800101": "PAGAMENTO DE EMPRESTIMO",
    # "4800102": "EMPRESTIMO (PAGAMENTO)",
    # "1000301": "PRESTACAO DO IMOVEL",
    # "4801602": "CREDITO EDUCATIVO (PAGAMENTO)",
    # "4801603": "PAGAMENTO DE CREDITO EDUCATIVO",
}

dividas_inadimplencia = {
    "4802201": "PAGAMENTO DE TITULO PROTESTADO",
    "5501602": "PENHORA (EMPRESTIMO)",
}

dividas_atraso = {
    "1000201": "ADICIONAIS DO ALUGUEL (JUROS, MULTA)",
    "1000801": "ADICIONAIS DE PRESTACAO DO IMOVEL",
    "1000901": "ADICIONAIS DE ALUGUEL DE GARAGEM",
    "1001001": "ADICIONAIS DE CONDOMINIO",
    "1001101": "ADICIONAIS DE IPTU",
    "1001201": "ADICIONAIS DE IPTR",
    "1203201": "JUROS E MULTA DE ENERGIA ELETRICA",
    "1203301": "JUROS E MULTA DE CONTA DE AGUA",
}


dividas = {}
dividas.update(dividas_custo)
dividas.update(dividas_amortizacao)
dividas.update(dividas_inadimplencia)
dividas.update(dividas_atraso)  # opcional

codigos_divida = list(dividas.keys())

print(f"✓ Códigos de dívida definidos: {len(codigos_divida)} categorias")

✓ Códigos de dívida definidos: 14 categorias


## 5. Agregação por UC e construção das variáveis analíticas

### Sobre a UC (Unidade de Consumo)

A **UC** é o grupo de pessoas que compartilha o mesmo domicílio e as mesmas
despesas — equivale à família pesquisada. Cada UC é identificada pela chave
composta `(COD_UPA, NUM_DOM, NUM_UC)`.

### Decisões de agregação

| Variável | Critério | Justificativa |
|---|---|---|
| `escolaridade_media` | Média de `ANOS_ESTUDO` | Captura o capital humano médio da UC |
| `nivel_instrucao_modal` | **Moda** de `NIVEL_INSTRUCAO` | Nível categorizado — média não faz sentido semântico em escala ordinal |
| `idade_media` | Média de `V0403` | Controla ciclo de vida da UC |
| `sexo_chefe` | **Primeiro** registro (assumido como responsável) | A POF ordena o responsável na posição 1 dentro da UC |
| `renda_total` | **Primeiro** valor por UC | `RENDA_TOTAL` é variável da UC, idêntica para todos os membros; média distorceria |
| `total_divida` | **Soma** de `V8000` filtrado | Acumula todos os encargos financeiros pagos pela UC no período |

### Novas variáveis derivadas (razão dívida/renda)

O foco analítico correto **não é o valor absoluto da dívida**, pois famílias mais
educadas têm renda mais alta e naturalmente tomam empréstimos maiores. O que
importa é **quanto da renda vai para encargos financeiros**:

- `razao_divida_renda` = `total_divida / renda_total` — comprometimento relativo
- `tem_divida` = 1 se `total_divida > 0` — indicador binário (para regressão logística)
- `log_divida_condicional` = `log(total_divida)` apenas para UCs com `total_divida > 0` — para o segundo estágio

In [ ]:
import numpy as np

morador    = dataframes_pof['MORADOR']
despesa_ind = dataframes_pof['DESPESA_INDIVIDUAL']
despesa_col = dataframes_pof['DESPESA_COLETIVA']

# ──────────────────────────────────────────────────────────────────────────────
# 5.1  Agrega variáveis individuais do morador → nível UC
# ──────────────────────────────────────────────────────────────────────────────
# Notas:
#   - ESCOLARIDADE (ANOS_ESTUDO): calcula 5 métodos (min, mediana, moda, max, média)
#   - NÍVEL INSTRUÇÃO: calcula 5 métodos (min, mediana, moda, max, média)
#   - Todos os valores são arredondados para inteiros após agregação
#   - RENDA_TOTAL: valor único por UC (todos os membros têm o mesmo) → first()
#   - sexo_chefe: a POF registra o responsável em primeiro lugar dentro da UC

morador_agg = morador.group_by(['COD_UPA', 'NUM_DOM', 'NUM_UC', 'UF']).agg([
    # ESCOLARIDADE — 5 métodos, arredondados
    pl.col('ANOS_ESTUDO').cast(pl.Float64).min().round().alias('escolaridade_min'),
    pl.col('ANOS_ESTUDO').cast(pl.Float64).median().round().alias('escolaridade_mediana'),
    pl.col('ANOS_ESTUDO').cast(pl.Float64).mean().round().alias('escolaridade_media'),
    pl.col('ANOS_ESTUDO').cast(pl.Float64).max().round().alias('escolaridade_max'),
    pl.col('ANOS_ESTUDO').cast(pl.Float64).mode().first().round().alias('escolaridade_mode'),
    
    # NÍVEL INSTRUÇÃO — 5 métodos, arredondados
    pl.col('NIVEL_INSTRUCAO').cast(pl.Float64).min().round().alias('nivel_min'),
    pl.col('NIVEL_INSTRUCAO').cast(pl.Float64).median().round().alias('nivel_mediana'),
    pl.col('NIVEL_INSTRUCAO').cast(pl.Float64).mean().round().alias('nivel_media'),
    pl.col('NIVEL_INSTRUCAO').cast(pl.Float64).max().round().alias('nivel_max'),
    pl.col('NIVEL_INSTRUCAO').cast(pl.Float64).mode().first().round().alias('nivel_mode'),
    
    # IDADE — 4 métodos, arredondados
    pl.col('V0403').cast(pl.Float64).min().round().alias('idade_minima'),
    pl.col('V0403').cast(pl.Float64).median().round().alias('idade_mediana'),
    pl.col('V0403').cast(pl.Float64).mean().round().alias('idade_media'),
    pl.col('V0403').cast(pl.Float64).max().round().alias('idade_maxima'),
    
    # Outras variáveis
    pl.col('V0404').first().alias('sexo_chefe'),          # 1=Homem, 2=Mulher
    pl.col('RENDA_TOTAL').cast(pl.Float64).first().alias('renda_total'),  # idêntica para todos na UC
    pl.col('ANOS_ESTUDO').cast(pl.Float64).count().alias('n_membros'),
])

# ──────────────────────────────────────────────────────────────────────────────
# 5.1b Agrega APENAS ADULTOS COM RENDA (V0403 > 18) & (V0407 == 1) — mesmas variáveis com sufixo _adultos
# ──────────────────────────────────────────────────────────────────────────────
morador_adultos = morador.filter(
    (pl.col('V0403').cast(pl.Float64) >= 18) & (pl.col('V0407').cast(pl.Float64) == 1)
)

morador_agg_adultos = morador_adultos.group_by(['COD_UPA', 'NUM_DOM', 'NUM_UC']).agg([
    # ESCOLARIDADE — 5 métodos, arredondados (apenas adultos)
    pl.col('ANOS_ESTUDO').cast(pl.Float64).min().round().alias('escolaridade_min_adultos'),
    pl.col('ANOS_ESTUDO').cast(pl.Float64).median().round().alias('escolaridade_mediana_adultos'),
    pl.col('ANOS_ESTUDO').cast(pl.Float64).mean().round().alias('escolaridade_media_adultos'),
    pl.col('ANOS_ESTUDO').cast(pl.Float64).max().round().alias('escolaridade_max_adultos'),
    pl.col('ANOS_ESTUDO').cast(pl.Float64).mode().first().round().alias('escolaridade_mode_adultos'),
    
    # NÍVEL INSTRUÇÃO — 5 métodos, arredondados (apenas adultos)
    pl.col('NIVEL_INSTRUCAO').cast(pl.Float64).min().round().alias('nivel_min_adultos'),
    pl.col('NIVEL_INSTRUCAO').cast(pl.Float64).median().round().alias('nivel_mediana_adultos'),
    pl.col('NIVEL_INSTRUCAO').cast(pl.Float64).mean().round().alias('nivel_media_adultos'),
    pl.col('NIVEL_INSTRUCAO').cast(pl.Float64).max().round().alias('nivel_max_adultos'),
    pl.col('NIVEL_INSTRUCAO').cast(pl.Float64).mode().first().round().alias('nivel_mode_adultos'),
    
    # IDADE — 4 métodos, arredondados (apenas adultos)
    pl.col('V0403').cast(pl.Float64).min().round().alias('idade_minima_adultos'),
    pl.col('V0403').cast(pl.Float64).median().round().alias('idade_mediana_adultos'),
    pl.col('V0403').cast(pl.Float64).mean().round().alias('idade_media_adultos'),
    pl.col('V0403').cast(pl.Float64).max().round().alias('idade_maxima_adultos'),
    
    # Contador de adultos na UC
    pl.col('ANOS_ESTUDO').cast(pl.Float64).count().alias('n_adultos'),
])

# Join das agregações de adultos com morador_agg
# morador_agg = morador_agg.join(
#     morador_agg_adultos,
#     on=['COD_UPA', 'NUM_DOM', 'NUM_UC'],
#     how='left'
# ).fill_null(-999)  # Valor sentinel para UCs sem adultos


# ──────────────────────────────────────────────────────────────────────────────
# 5.2  Filtra registros de dívida e agrega por UC
# ──────────────────────────────────────────────────────────────────────────────

dividas_ind = despesa_ind.filter(pl.col('V9001').is_in(codigos_divida))
dividas_col = despesa_col.filter(pl.col('V9001').is_in(codigos_divida))

# V800: Valor em reais (R$), considerando os centavos, da despesa/aquisição realizada
# pelo informante no período de referência da pesquisa. Nos casos de valores ignorados
# (não determinados), este campo está preenchido com 9999999.99
dividas_ind = dividas_ind.filter(pl.col('V8000') < 99999.9999)
dividas_col = dividas_col.filter(pl.col('V8000') < 99999.9999)

print(f"✓ Registros de dívida em DESPESA_INDIVIDUAL: {dividas_ind.shape[0]:,}")
print(f"✓ Registros de dívida em DESPESA_COLETIVA:   {dividas_col.shape[0]:,}")

# Soma dívida individual por UC
divida_agg_ind = dividas_ind.group_by(['COD_UPA', 'NUM_DOM', 'NUM_UC']).agg(
    pl.col('V8000_DEFLA').cast(pl.Float64).sum().alias('total_divida_ind'),
    pl.col('RENDA_TOTAL').cast(pl.Float64).first().alias('renda_total_ind')
)

# Soma dívida coletiva por UC (quando existir)
if dividas_col.shape[0] > 0:
    divida_agg_col = dividas_col.group_by(['COD_UPA', 'NUM_DOM', 'NUM_UC']).agg(
        pl.col('V8000_DEFLA').cast(pl.Float64).sum().alias('total_divida_col'),
        pl.col('RENDA_TOTAL').cast(pl.Float64).first().alias('renda_total_col')
    )
else:
    # Cria tabela vazia com as mesmas colunas para o join não quebrar
    divida_agg_col = pl.DataFrame({
        'COD_UPA': pl.Series([], dtype=pl.Utf8),
        'NUM_DOM': pl.Series([], dtype=pl.Utf8),
        'NUM_UC':  pl.Series([], dtype=pl.Utf8),
        'total_divida_col': pl.Series([], dtype=pl.Float64),
        'renda_total_col': pl.Series([], dtype=pl.Float64),
    })

# ──────────────────────────────────────────────────────────────────────────────
# 5.3  Junta tudo
# ──────────────────────────────────────────────────────────────────────────────
df_final = (
    morador_agg
    .join(divida_agg_ind, on=['COD_UPA', 'NUM_DOM', 'NUM_UC'], how='left')
    .join(divida_agg_col, on=['COD_UPA', 'NUM_DOM', 'NUM_UC'], how='left')
    .fill_null(0)
)

# Dívida total = individual + coletiva
df_final = df_final.with_columns(
    (pl.col('total_divida_ind') + pl.col('total_divida_col')).alias('total_divida')
)

# ──────────────────────────────────────────────────────────────────────────────
# 5.4  Filtro de qualidade: remove UCs com renda zero
#      (renda=0 indica dado faltante ou erro de registro, não pobreza extrema
#      — UCs com renda muito baixa mas positiva são mantidas)
#
#  ATENÇÃO: NÃO filtramos por escolaridade=0. UCs sem instrução são
#  exatamente o grupo de interesse comparativo — removê-las introduziria
#  viés de seleção.
# ──────────────────────────────────────────────────────────────────────────────
# df_final = df_final.filter(pl.col('renda_total') > 0)

# ──────────────────────────────────────────────────────────────────────────────
# 5.5  Variáveis derivadas
# ──────────────────────────────────────────────────────────────────────────────
df_final = df_final.with_columns([
    # Razão dívida/renda — variável central da análise
    (pl.col('total_divida') / pl.col('renda_total')).alias('razao_divida_renda'),

    # Indicador binário: a UC tem alguma dívida?
    (pl.col('total_divida') > 0).cast(pl.Int8).alias('tem_divida'),
])

print(f"\n✅ Dataset final: {df_final.shape[0]:,} UCs com renda > 0")
print(f"   UCs com dívida > 0: {df_final.filter(pl.col('total_divida') > 0).shape[0]:,} "
      f"({df_final.filter(pl.col('total_divida') > 0).shape[0]/df_final.shape[0]*100:.1f}%)")
print(f"\n📊 Colunas em df_final (com 10 métodos de agregação arredondados):")
print(f"   {df_final.columns}")


# ──────────────────────────────────────────────────────────────────────────────
# 5.6  Cria df_final_adultos — mesma estrutura, apenas com dados de adultos (>18)
# ──────────────────────────────────────────────────────────────────────────────
df_final_adultos = (
    morador_agg_adultos
    .join(divida_agg_ind, on=['COD_UPA', 'NUM_DOM', 'NUM_UC'], how='left')
    .join(divida_agg_col, on=['COD_UPA', 'NUM_DOM', 'NUM_UC'], how='left')
    .fill_null(0)
)

# Dívida total = individual + coletiva
df_final_adultos = df_final_adultos.with_columns(
    (pl.col('total_divida_ind') + pl.col('total_divida_col')).alias('total_divida')
)

# Junta com renda_total de df_final (que é comum a toda UC)
df_final_adultos = df_final_adultos.join(
    df_final.select(['COD_UPA', 'NUM_DOM', 'NUM_UC', 'renda_total']),
    on=['COD_UPA', 'NUM_DOM', 'NUM_UC'],
    how='left'
)

# Filtro de qualidade: remove UCs com renda zero
# df_final_adultos = df_final_adultos.filter(pl.col('renda_total') > 0)

# Variáveis derivadas
df_final_adultos = df_final_adultos.with_columns([
    (pl.col('total_divida') / pl.col('renda_total')).alias('razao_divida_renda'),
    (pl.col('total_divida') > 0).cast(pl.Int8).alias('tem_divida'),
])

print(f"\n✅ Dataset final (apenas adultos): {df_final_adultos.shape[0]:,} UCs com renda > 0 e com adultos")
print(f"   UCs com dívida > 0: {df_final_adultos.filter(pl.col('total_divida') > 0).shape[0]:,} "
      f"({df_final_adultos.filter(pl.col('total_divida') > 0).shape[0]/df_final_adultos.shape[0]*100:.1f}%)")
print(f"\n📊 Colunas em df_final_adultos:")
print(f"   {df_final_adultos.columns}")


✓ Registros de dívida em DESPESA_INDIVIDUAL: 4,393
✓ Registros de dívida em DESPESA_COLETIVA:   117

✅ Dataset final: 58,039 UCs com renda > 0
   UCs com dívida > 0: 3,831 (6.6%)

📊 Colunas em df_final (com 10 métodos de agregação arredondados):
   ['COD_UPA', 'NUM_DOM', 'NUM_UC', 'UF', 'escolaridade_min', 'escolaridade_mediana', 'escolaridade_media', 'escolaridade_max', 'escolaridade_mode', 'nivel_min', 'nivel_mediana', 'nivel_media', 'nivel_max', 'nivel_mode', 'idade_minima', 'idade_mediana', 'idade_media', 'idade_maxima', 'sexo_chefe', 'renda_total', 'n_membros', 'escolaridade_min_adultos', 'escolaridade_mediana_adultos', 'escolaridade_media_adultos', 'escolaridade_max_adultos', 'escolaridade_mode_adultos', 'nivel_min_adultos', 'nivel_mediana_adultos', 'nivel_media_adultos', 'nivel_max_adultos', 'nivel_mode_adultos', 'idade_minima_adultos', 'idade_mediana_adultos', 'idade_media_adultos', 'idade_maxima_adultos', 'n_adultos', 'total_divida_ind', 'renda_total_ind', 'total_divida_col', 

In [63]:
df_final_adultos.filter(pl.col('renda_total_col')>0)

COD_UPA,NUM_DOM,NUM_UC,escolaridade_min_adultos,escolaridade_mediana_adultos,escolaridade_media_adultos,escolaridade_max_adultos,escolaridade_mode_adultos,nivel_min_adultos,nivel_mediana_adultos,nivel_media_adultos,nivel_max_adultos,nivel_mode_adultos,idade_minima_adultos,idade_mediana_adultos,idade_media_adultos,idade_maxima_adultos,n_adultos,total_divida_ind,renda_total_ind,total_divida_col,renda_total_col,total_divida,renda_total,razao_divida_renda,tem_divida
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,i8
"""230105810""","""4""","""1""",10.0,11.0,11.0,12.0,10.0,4.0,4.0,4.0,5.0,4.0,21.0,28.0,28.0,34.0,2,0.0,0.0,0.8648,55.325,0.8648,55.325,0.015631,1
"""220039944""","""9""","""1""",0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,37.0,37.0,37.0,37.0,1,0.0,0.0,0.0056,6.8895,0.0056,6.8895,0.000813,1
"""410124161""","""1""","""1""",0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,27.0,27.0,27.0,27.0,1,0.0,0.0,1.0955,25.311,1.0955,25.311,0.043282,1
"""310032653""","""1""","""1""",7.0,8.0,8.0,9.0,9.0,2.0,2.0,2.0,3.0,3.0,26.0,29.0,29.0,32.0,2,0.0,0.0,0.3384,37.0816,0.3384,37.0816,0.009126,1
"""220048949""","""1""","""1""",16.0,16.0,16.0,16.0,16.0,7.0,7.0,7.0,7.0,7.0,40.0,40.0,40.0,40.0,1,0.0,0.0,0.0083,49.8167,0.0083,49.8167,0.000167,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""220011849""","""1""","""1""",5.0,10.0,10.0,16.0,5.0,2.0,4.0,4.0,7.0,2.0,40.0,47.0,47.0,54.0,2,0.0,0.0,0.0111,35.128,0.0111,35.128,0.000316,1
"""220000939""","""14""","""1""",0.0,3.0,5.0,12.0,12.0,1.0,2.0,3.0,5.0,1.0,22.0,62.0,51.0,69.0,3,0.0,0.0,0.0136,28.2305,0.0136,28.2305,0.000482,1
"""280034214""","""5""","""1""",12.0,14.0,14.0,16.0,16.0,5.0,6.0,6.0,7.0,7.0,22.0,32.0,32.0,42.0,2,0.599,50.5246,0.1,50.5246,0.699,50.5246,0.013835,1


In [64]:
df_final.filter(pl.col('total_divida_col')>0)

COD_UPA,NUM_DOM,NUM_UC,UF,escolaridade_min,escolaridade_mediana,escolaridade_media,escolaridade_max,escolaridade_mode,nivel_min,nivel_mediana,nivel_media,nivel_max,nivel_mode,idade_minima,idade_mediana,idade_media,idade_maxima,sexo_chefe,renda_total,n_membros,escolaridade_min_adultos,escolaridade_mediana_adultos,escolaridade_media_adultos,escolaridade_max_adultos,escolaridade_mode_adultos,nivel_min_adultos,nivel_mediana_adultos,nivel_media_adultos,nivel_max_adultos,nivel_mode_adultos,idade_minima_adultos,idade_mediana_adultos,idade_media_adultos,idade_maxima_adultos,n_adultos,total_divida_ind,renda_total_ind,total_divida_col,renda_total_col,total_divida,razao_divida_renda,tem_divida
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,i8
"""330028112""","""12""","""1""","""33""",6.0,7.0,7.0,8.0,6.0,2.0,2.0,2.0,2.0,2.0,61.0,62.0,62.0,63.0,"""2""",34.2845,2,6.0,7.0,7.0,8.0,8.0,2.0,2.0,2.0,2.0,2.0,61.0,62.0,62.0,63.0,2,0.0,0.0,0.06,34.2845,0.06,0.00175,1
"""350135478""","""3""","""1""","""35""",6.0,9.0,9.0,13.0,9.0,2.0,3.0,4.0,6.0,6.0,29.0,50.0,44.0,52.0,"""1""",63.4953,3,6.0,9.0,9.0,13.0,6.0,2.0,3.0,4.0,6.0,2.0,29.0,50.0,44.0,52.0,3,0.0,0.0,0.818,63.4953,0.818,0.012883,1
"""240018574""","""6""","""1""","""24""",6.0,10.0,9.0,12.0,12.0,2.0,3.0,3.0,5.0,2.0,14.0,26.0,26.0,39.0,"""1""",24.2173,4,6.0,6.0,6.0,6.0,6.0,2.0,2.0,2.0,2.0,2.0,37.0,37.0,37.0,37.0,1,0.0,0.0,0.4983,24.2173,0.4983,0.020576,1
"""500038744""","""10""","""1""","""50""",4.0,8.0,8.0,12.0,12.0,2.0,4.0,4.0,5.0,5.0,20.0,32.0,32.0,44.0,"""2""",13.903,2,4.0,8.0,8.0,12.0,4.0,2.0,4.0,4.0,5.0,5.0,20.0,32.0,32.0,44.0,2,0.0,0.0,0.0783,13.903,0.0783,0.005632,1
"""350632369""","""8""","""1""","""35""",0.0,1.0,2.0,4.0,1.0,1.0,2.0,2.0,2.0,2.0,7.0,13.0,26.0,52.0,"""1""",22.8741,5,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,52.0,52.0,52.0,52.0,1,0.0,0.0,0.2754,22.8741,0.2754,0.01204,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""330015380""","""12""","""1""","""33""",13.0,13.0,13.0,13.0,13.0,5.0,5.0,5.0,5.0,5.0,82.0,82.0,82.0,82.0,"""2""",12.1126,1,13.0,13.0,13.0,13.0,13.0,5.0,5.0,5.0,5.0,5.0,82.0,82.0,82.0,82.0,1,0.0,0.0,0.7418,12.1126,0.7418,0.061242,1
"""220048949""","""5""","""1""","""22""",2.0,4.0,4.0,5.0,4.0,2.0,2.0,2.0,2.0,2.0,67.0,85.0,82.0,95.0,"""2""",35.9003,3,2.0,4.0,4.0,5.0,5.0,2.0,2.0,2.0,2.0,2.0,67.0,85.0,82.0,95.0,3,0.0,0.0,0.0087,35.9003,0.0087,0.000242,1
"""220039944""","""4""","""1""","""22""",0.0,0.0,5.0,16.0,0.0,1.0,1.0,3.0,7.0,1.0,45.0,73.0,64.0,73.0,"""1""",22.4611,3,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,73.0,73.0,73.0,73.0,2,0.0,0.0,0.0869,22.4611,0.0869,0.003869,1


In [62]:
df_final_adultos.filter(pl.col('renda_total_col')>0).select(['renda_total_ind', 'renda_total_col', 'renda_total']).describe()

statistic,renda_total_ind,renda_total_col,renda_total
str,f64,f64,f64
"""count""",104.0,104.0,104.0
"""null_count""",0.0,0.0,0.0
"""mean""",15.356561,56.288631,56.288631
"""std""",48.026911,74.530662,74.530662
"""min""",0.0,6.2631,6.2631
"""25%""",0.0,20.308,20.308
"""50%""",0.0,35.128,35.128
"""75%""",0.0,67.8034,67.8034
"""max""",376.8714,603.4138,603.4138


## 6. Validação das dívidas agregadas

In [44]:
print("=" * 70)
print("VALIDAÇÃO: estatísticas descritivas das variáveis-chave")
print("=" * 70)

import numpy as np
import pandas as pd

# Resumo das colunas agregadas em df_final
print("\n📊 Escolaridade — 5 métodos (todos arredondados para inteiros):")
print("   escolaridade_min, escolaridade_mediana, escolaridade_media, escolaridade_max, escolaridade_mode")

print("\n📊 Nível Instrução — 5 métodos (todos arredondados para inteiros):")
print("   nivel_min, nivel_mediana, nivel_media, nivel_max, nivel_mode")

# ──────────────────────────────────────────────────────────────────────────────
# Estatísticas: TODOS os membros da UC
# ──────────────────────────────────────────────────────────────────────────────
stats = df_final.select([
    pl.col('total_divida').mean().alias('divida_media'),
    pl.col('total_divida').median().alias('divida_mediana'),
    pl.col('total_divida').max().alias('divida_max'),
    pl.col('razao_divida_renda').mean().alias('razao_media'),
    pl.col('razao_divida_renda').median().alias('razao_mediana'),
    pl.col('renda_total').mean().alias('renda_media'),
])
print("\n📈 Sumário geral de dívida (TODOS os membros, n={:,}):".format(df_final.shape[0]))
display(stats)

# Amostra do df_final com as novas colunas de escolaridade e nível instrução
print("\n📋 Amostra de df_final — TODOS (primeiras 5 UCs):")
display(df_final.select(['COD_UPA', 'NUM_DOM', 'NUM_UC', 
                        'escolaridade_min', 'escolaridade_mediana', 'escolaridade_media', 'escolaridade_max', 'escolaridade_mode',
                        'nivel_min', 'nivel_mediana', 'nivel_media', 'nivel_max', 'nivel_mode',
                        'renda_total', 'total_divida']).head(5))

# ──────────────────────────────────────────────────────────────────────────────
# Estatísticas: APENAS adultos (>18 anos)
# ──────────────────────────────────────────────────────────────────────────────
stats_adultos = df_final_adultos.select([
    pl.col('total_divida').mean().alias('divida_media'),
    pl.col('total_divida').median().alias('divida_mediana'),
    pl.col('total_divida').max().alias('divida_max'),
    pl.col('razao_divida_renda').mean().alias('razao_media'),
    pl.col('razao_divida_renda').median().alias('razao_mediana'),
    pl.col('renda_total').mean().alias('renda_media'),
])
print("\n\n📈 Sumário geral de dívida (APENAS ADULTOS COM RENDA, n={:,}):".format(df_final_adultos.shape[0]))
display(stats_adultos)

# Amostra do df_final_adultos
print("\n📋 Amostra de df_final_adultos (primeiras 5 UCs):")
display(df_final_adultos.select(['COD_UPA', 'NUM_DOM', 'NUM_UC', 
                                  'escolaridade_min_adultos', 'escolaridade_mediana_adultos', 'escolaridade_media_adultos', 
                                  'escolaridade_max_adultos', 'escolaridade_mode_adultos',
                                  'nivel_min_adultos', 'nivel_mediana_adultos', 'nivel_media_adultos', 
                                  'nivel_max_adultos', 'nivel_mode_adultos',
                                  'renda_total', 'total_divida', 'n_adultos']).head(5))

print("\n✅ Dados preparados para ambos datasets — próximo passo: sessão 6 (visualização)")

VALIDAÇÃO: estatísticas descritivas das variáveis-chave

📊 Escolaridade — 5 métodos (todos arredondados para inteiros):
   escolaridade_min, escolaridade_mediana, escolaridade_media, escolaridade_max, escolaridade_mode

📊 Nível Instrução — 5 métodos (todos arredondados para inteiros):
   nivel_min, nivel_mediana, nivel_media, nivel_max, nivel_mode

📈 Sumário geral de dívida (TODOS os membros, n=58,039):


divida_media,divida_mediana,divida_max,razao_media,razao_mediana,renda_media
f64,f64,f64,f64,f64,f64
0.162538,0.0,143.3265,NaN,0.0,46.702687



📋 Amostra de df_final — TODOS (primeiras 5 UCs):


COD_UPA,NUM_DOM,NUM_UC,escolaridade_min,escolaridade_mediana,escolaridade_media,escolaridade_max,escolaridade_mode,nivel_min,nivel_mediana,nivel_media,nivel_max,nivel_mode,renda_total,total_divida
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""290067091""","""10""","""1""",9.0,9.0,9.0,9.0,9.0,3.0,3.0,3.0,3.0,3.0,11.9009,0.0
"""500016747""","""10""","""1""",0.0,11.0,10.0,16.0,16.0,1.0,4.0,4.0,7.0,7.0,105.738,0.0
"""420077838""","""10""","""1""",5.0,5.0,5.0,5.0,5.0,2.0,2.0,2.0,2.0,2.0,38.2443,0.0
"""530038222""","""4""","""1""",5.0,9.0,9.0,12.0,5.0,2.0,3.0,3.0,5.0,2.0,12.014,0.0
"""230052507""","""1""","""1""",15.0,15.0,15.0,16.0,15.0,6.0,6.0,6.0,7.0,6.0,11.7996,0.0




📈 Sumário geral de dívida (APENAS ADULTOS COM RENDA, n=57,966):


divida_media,divida_mediana,divida_max,razao_media,razao_mediana,renda_media
f64,f64,f64,f64,f64,f64
0.162743,0.0,143.3265,NaN,0.0,46.748908



📋 Amostra de df_final_adultos (primeiras 5 UCs):


COD_UPA,NUM_DOM,NUM_UC,escolaridade_min_adultos,escolaridade_mediana_adultos,escolaridade_media_adultos,escolaridade_max_adultos,escolaridade_mode_adultos,nivel_min_adultos,nivel_mediana_adultos,nivel_media_adultos,nivel_max_adultos,nivel_mode_adultos,renda_total,total_divida,n_adultos
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""430164390""","""10""","""1""",6.0,10.0,10.0,13.0,6.0,2.0,4.0,4.0,6.0,6.0,29.7872,0.0,2
"""150084303""","""7""","""1""",12.0,12.0,12.0,12.0,12.0,5.0,5.0,5.0,5.0,5.0,10.9371,0.0,1
"""310139778""","""4""","""1""",7.0,12.0,10.0,12.0,12.0,2.0,5.0,4.0,5.0,5.0,36.5653,0.0,3
"""330208798""","""9""","""1""",12.0,12.0,12.0,12.0,12.0,5.0,5.0,5.0,5.0,5.0,16.0922,0.0,1
"""230012447""","""3""","""1""",7.0,8.0,8.0,9.0,7.0,2.0,2.0,2.0,3.0,2.0,23.9613,0.0,2



✅ Dados preparados para ambos datasets — próximo passo: sessão 6 (visualização)


### Qual método de nível instrução tem maior relação com renda?

In [45]:
# Análise de correlação: qual método (escolaridade ou nível) tem maior relação com renda?
# Comparando TODOS os membros vs APENAS ADULTOS

# Define os métodos de cada variável
nivel_cols_todos = ['nivel_min', 'nivel_mediana', 'nivel_media', 'nivel_max', 'nivel_mode']
escolaridade_cols_todos = ['escolaridade_min', 'escolaridade_mediana', 'escolaridade_media', 'escolaridade_max', 'escolaridade_mode']

nivel_cols_adultos = ['nivel_min_adultos', 'nivel_mediana_adultos', 'nivel_media_adultos', 'nivel_max_adultos', 'nivel_mode_adultos']
escolaridade_cols_adultos = ['escolaridade_min_adultos', 'escolaridade_mediana_adultos', 'escolaridade_media_adultos', 'escolaridade_max_adultos', 'escolaridade_mode_adultos']

renda_col = 'renda_total'

# ──────────────────────────────────────────────────────────────────────────────
# Análise 1: TODOS os membros da UC
# ──────────────────────────────────────────────────────────────────────────────
pdf_todos = df_final.to_pandas()

correlacoes_nivel_todos = {}
for col in nivel_cols_todos:
    mask = (pdf_todos[col].notna()) & (pdf_todos[renda_col].notna())
    corr = pdf_todos.loc[mask, col].corr(pdf_todos.loc[mask, renda_col])
    correlacoes_nivel_todos[col] = corr

correlacoes_escolaridade_todos = {}
for col in escolaridade_cols_todos:
    mask = (pdf_todos[col].notna()) & (pdf_todos[renda_col].notna())
    corr = pdf_todos.loc[mask, col].corr(pdf_todos.loc[mask, renda_col])
    correlacoes_escolaridade_todos[col] = corr

# Ordena
nivel_ordenado_todos = dict(sorted(correlacoes_nivel_todos.items(), key=lambda x: x[1], reverse=True))
escolaridade_ordenado_todos = dict(sorted(correlacoes_escolaridade_todos.items(), key=lambda x: x[1], reverse=True))

# ──────────────────────────────────────────────────────────────────────────────
# Análise 2: APENAS ADULTOS (>18 anos)
# ──────────────────────────────────────────────────────────────────────────────
pdf_adultos = df_final_adultos.to_pandas()

correlacoes_nivel_adultos = {}
for col in nivel_cols_adultos:
    mask = (pdf_adultos[col].notna()) & (pdf_adultos[renda_col].notna())
    corr = pdf_adultos.loc[mask, col].corr(pdf_adultos.loc[mask, renda_col])
    correlacoes_nivel_adultos[col] = corr

correlacoes_escolaridade_adultos = {}
for col in escolaridade_cols_adultos:
    mask = (pdf_adultos[col].notna()) & (pdf_adultos[renda_col].notna())
    corr = pdf_adultos.loc[mask, col].corr(pdf_adultos.loc[mask, renda_col])
    correlacoes_escolaridade_adultos[col] = corr

# Ordena
nivel_ordenado_adultos = dict(sorted(correlacoes_nivel_adultos.items(), key=lambda x: x[1], reverse=True))
escolaridade_ordenado_adultos = dict(sorted(correlacoes_escolaridade_adultos.items(), key=lambda x: x[1], reverse=True))

# ──────────────────────────────────────────────────────────────────────────────
# Exibe comparação
# ──────────────────────────────────────────────────────────────────────────────
print("=" * 80)
print("📊 CORRELAÇÃO (Pearson) com renda_total: COMPARAÇÃO TODOS vs ADULTOS")
print("=" * 80)

print("\n" + "─" * 80)
print("NÍVEL INSTRUÇÃO — 5 métodos")
print("─" * 80)
print(f"{'Método':<15} {'TODOS':<20} {'APENAS ADULTOS':<20} {'Diferença':<15}")
print("─" * 80)

for metodo_todos, metodo_adultos in zip(nivel_ordenado_todos.keys(), nivel_ordenado_adultos.keys()):
    corr_todos = correlacoes_nivel_todos[metodo_todos]
    corr_adultos = correlacoes_nivel_adultos[metodo_adultos]
    label = metodo_todos.replace('nivel_', '').upper()
    diff = corr_adultos - corr_todos
    print(f"{label:<15} {corr_todos:>8.4f}        {corr_adultos:>8.4f}           {diff:>+8.4f}")

print("\n" + "─" * 80)
print("ESCOLARIDADE — 5 métodos")
print("─" * 80)
print(f"{'Método':<15} {'TODOS':<20} {'APENAS ADULTOS':<20} {'Diferença':<15}")
print("─" * 80)

for metodo_todos, metodo_adultos in zip(escolaridade_ordenado_todos.keys(), escolaridade_ordenado_adultos.keys()):
    corr_todos = correlacoes_escolaridade_todos[metodo_todos]
    corr_adultos = correlacoes_escolaridade_adultos[metodo_adultos]
    label = metodo_todos.replace('escolaridade_', '').upper()
    diff = corr_adultos - corr_todos
    print(f"{label:<15} {corr_todos:>8.4f}        {corr_adultos:>8.4f}           {diff:>+8.4f}")

# ──────────────────────────────────────────────────────────────────────────────
# Ranking geral
# ──────────────────────────────────────────────────────────────────────────────
melhor_nivel_todos = list(nivel_ordenado_todos.items())[0]
melhor_escolaridade_todos = list(escolaridade_ordenado_todos.items())[0]
melhor_nivel_adultos = list(nivel_ordenado_adultos.items())[0]
melhor_escolaridade_adultos = list(escolaridade_ordenado_adultos.items())[0]

print("\n" + "=" * 80)
print("🎯 RANKING: Melhor método em cada grupo")
print("=" * 80)

print("\nTODOS os membros da UC (n={:,}):".format(df_final.shape[0]))
if melhor_nivel_todos[1] > melhor_escolaridade_todos[1]:
    print(f"  1️⃣  {melhor_nivel_todos[0].replace('nivel_', '').upper():15} (NÍVEL)        → r = {melhor_nivel_todos[1]:.4f}")
    print(f"  2️⃣  {melhor_escolaridade_todos[0].replace('escolaridade_', '').upper():15} (ESCOLARIDADE) → r = {melhor_escolaridade_todos[1]:.4f}")
else:
    print(f"  1️⃣  {melhor_escolaridade_todos[0].replace('escolaridade_', '').upper():15} (ESCOLARIDADE) → r = {melhor_escolaridade_todos[1]:.4f}")
    print(f"  2️⃣  {melhor_nivel_todos[0].replace('nivel_', '').upper():15} (NÍVEL)        → r = {melhor_nivel_todos[1]:.4f}")

print("\nAPENAS ADULTOS >18 (n={:,}):".format(df_final_adultos.shape[0]))
if melhor_nivel_adultos[1] > melhor_escolaridade_adultos[1]:
    print(f"  1️⃣  {melhor_nivel_adultos[0].replace('nivel_', '_').replace('_adultos', '').upper():15} (NÍVEL)        → r = {melhor_nivel_adultos[1]:.4f}")
    print(f"  2️⃣  {melhor_escolaridade_adultos[0].replace('escolaridade_', '_').replace('_adultos', '').upper():15} (ESCOLARIDADE) → r = {melhor_escolaridade_adultos[1]:.4f}")
else:
    print(f"  1️⃣  {melhor_escolaridade_adultos[0].replace('escolaridade_', '_').replace('_adultos', '').upper():15} (ESCOLARIDADE) → r = {melhor_escolaridade_adultos[1]:.4f}")
    print(f"  2️⃣  {melhor_nivel_adultos[0].replace('nivel_', '_').replace('_adultos', '').upper():15} (NÍVEL)        → r = {melhor_nivel_adultos[1]:.4f}")


📊 CORRELAÇÃO (Pearson) com renda_total: COMPARAÇÃO TODOS vs ADULTOS

────────────────────────────────────────────────────────────────────────────────
NÍVEL INSTRUÇÃO — 5 métodos
────────────────────────────────────────────────────────────────────────────────
Método          TODOS                APENAS ADULTOS       Diferença      
────────────────────────────────────────────────────────────────────────────────
MEDIA             0.3242          0.3253            +0.0011
MEDIANA           0.3191          0.3235            +0.0043
MAX               0.3084          0.3208            +0.0124
MODE              0.2933          0.3029            +0.0096
MIN               0.2148          0.2816            +0.0667

────────────────────────────────────────────────────────────────────────────────
ESCOLARIDADE — 5 métodos
────────────────────────────────────────────────────────────────────────────────
Método          TODOS                APENAS ADULTOS       Diferença      
────────────────────────

## 7. Análise Descritiva

### Por que dívida/renda e não dívida absoluta?

Um fato estilizado bem documentado é que educação aumenta renda. Se analisarmos
apenas o **valor absoluto** da dívida, o efeito positivo pode simplesmente
refletir que famílias mais educadas têm mais renda e, portanto, mais capacidade
de tomar crédito e de pagar valores maiores de encargos — sem que o
**comprometimento relativo** da renda seja maior.

A razão `dívida / renda` é a variável economicamente mais relevante para avaliar
se famílias mais educadas são, proporcionalmente, mais ou menos endividadas.


In [ ]:
# Sessão 6: Visualizar distribuição de UCs por escolaridade/nível instrução
# Os valores JÁ ESTÃO agregados e arredondados em df_final
# Precisamos apenas contar quantas UCs tem cada valor por método

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

pdf = df_final.to_pandas()

# ─────────────────────────────────────────────────────────────────────────────
# ESCOLARIDADE: contar UCs por valor (já arredondado) para cada método
# ─────────────────────────────────────────────────────────────────────────────

esc_methods = {
    'escolaridade_min': 'mínimo',
    'escolaridade_mediana': 'mediana',
    'escolaridade_mode': 'moda',
    'escolaridade_max': 'máximo',
    'escolaridade_media': 'média'
}

esc_counts = {}
for col, label in esc_methods.items():
    # Converter para inteiro (já está arredondado)
    values = pdf[col].astype('Int64')
    # Contar frequências
    counts = values.value_counts().sort_index()
    esc_counts[label] = counts

# Criar figura com subplots para escolaridade
fig_esc, axes_esc = plt.subplots(1, 5, figsize=(18, 4))
fig_esc.suptitle('Distribuição de UCs por Escolaridade (anos) — Cada método de agregação\n' + 
                 'Valores JÁ arredondados na agregação', 
                 fontsize=13, fontweight='bold')

for idx, (label, counts) in enumerate(esc_counts.items()):
    ax = axes_esc[idx]
    counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(f'Método: {label}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Anos de Escolaridade')
    ax.set_ylabel('Número de UCs')
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# NÍVEL INSTRUÇÃO: contar UCs por valor (já arredondado) para cada método
# ─────────────────────────────────────────────────────────────────────────────

nivel_methods = {
    'nivel_min': 'mínimo',
    'nivel_mediana': 'mediana',
    'nivel_mode': 'moda',
    'nivel_max': 'máximo',
    'nivel_media': 'média'
}

nivel_counts = {}
for col, label in nivel_methods.items():
    # Converter para inteiro (já está arredondado)
    values = pdf[col].astype('Int64')
    # Contar frequências
    counts = values.value_counts().sort_index()
    nivel_counts[label] = counts

# Criar figura com subplots para nível instrução
fig_nivel, axes_nivel = plt.subplots(1, 5, figsize=(18, 4))
fig_nivel.suptitle('Distribuição de UCs por Nível de Instrução — Cada método de agregação\n' + 
                   'Valores JÁ arredondados na agregação', 
                   fontsize=13, fontweight='bold')

for idx, (label, counts) in enumerate(nivel_counts.items()):
    ax = axes_nivel[idx]
    counts.plot(kind='bar', ax=ax, color='coral', edgecolor='black', alpha=0.7)
    ax.set_title(f'Método: {label}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Nível de Instrução')
    ax.set_ylabel('Número de UCs')
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Resumo estatístico
# ─────────────────────────────────────────────────────────────────────────────

print("\n📊 ESCOLARIDADE — Resumo por método:")
for label in esc_methods.values():
    counts = esc_counts[label]
    print(f"\n  {label.upper()}:")
    print(f"    Intervalo: {int(counts.index.min())}-{int(counts.index.max())} anos")
    print(f"    Total UCs: {int(counts.sum())}")
    print(f"    Moda visual (maior pico): {int(counts.idxmax())} anos ({int(counts.max())} UCs)")

print("\n\n📊 NÍVEL INSTRUÇÃO — Resumo por método:")
for label in nivel_methods.values():
    counts = nivel_counts[label]
    print(f"\n  {label.upper()}:")
    print(f"    Intervalo: {int(counts.index.min())}-{int(counts.index.max())}")
    print(f"    Total UCs: {int(counts.sum())}")
    print(f"    Moda visual (maior pico): {int(counts.idxmax())} ({int(counts.max())} UCs)")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Converte para pandas para os gráficos e testes
df_pandas = df_final.to_pandas()

# ──────────────────────────────────────────────────────────────────────────────
# 7.1  Cria categorias de escolaridade
# ──────────────────────────────────────────────────────────────────────────────
df_pandas['escolaridade_cat'] = pd.cut(
    df_pandas['escolaridade_media'],
    bins=[-0.01, 4, 8, 11, 16],
    labels=['Nenhuma/Fundamental I (0-4)', 'Fundamental II (4-8)',
            'Médio (8-11)', 'Superior (11+)']
)

# ──────────────────────────────────────────────────────────────────────────────
# 7.2  Tabelas descritivas por nível de escolaridade
# ──────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("ESTATÍSTICAS DESCRITIVAS POR NÍVEL DE ESCOLARIDADE")
print("=" * 70)

tab_desc = df_pandas.groupby('escolaridade_cat', observed=True).agg(
    n_ucs=('total_divida', 'count'),
    # Dívida absoluta
    divida_media=('total_divida', 'mean'),
    divida_mediana=('total_divida', 'median'),
    # Razão dívida/renda  ← variável central
    razao_media=('razao_divida_renda', 'mean'),
    razao_mediana=('razao_divida_renda', 'median'),
    # Variáveis de contexto
    renda_media=('renda_total', 'mean'),
    idade_media=('idade_media', 'mean'),
    escolaridade_media=('escolaridade_media', 'mean'),
).round(3)

print(tab_desc.to_string())

# Percentual de UCs com dívida por escolaridade
print("\n\nPERCENTUAL DE FAMÍLIAS COM DÍVIDA POR NÍVEL DE ESCOLARIDADE")
print("-" * 60)
pct_divida = df_pandas.groupby('escolaridade_cat', observed=True)['tem_divida'].mean() * 100
print(pct_divida.round(1).to_string())

print("\n[NOTA] A razão dívida/renda é a variável central da análise.")
print("       Valores absolutos de dívida são apresentados apenas como referência.")

In [ ]:
debt_groups = {
    'todas_dividas': codigos_divida,
    'dividas_custo': list(dividas_custo.keys()),
    'dividas_amortizacao': list(dividas_amortizacao.keys()),
    'dividas_inadimplencia': list(dividas_inadimplencia.keys()),
    'dividas_atraso': list(dividas_atraso.keys()),
}
results = []
for name, codes in debt_groups.items():
    ind = despesa_ind.filter(pl.col('V9001').is_in(codes)).group_by(['COD_UPA','NUM_DOM','NUM_UC']).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_ind'))
    if despesa_col.shape[0] > 0:
        col = despesa_col.filter(pl.col('V9001').is_in(codes)).group_by(['COD_UPA','NUM_DOM','NUM_UC']).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_col'))
    else:
        col = pl.DataFrame({'COD_UPA': pl.Series([], dtype=pl.Utf8), 'NUM_DOM': pl.Series([], dtype=pl.Utf8), 'NUM_UC': pl.Series([], dtype=pl.Utf8), 'div_col': pl.Series([], dtype=pl.Float64)})
    merged = (morador_agg.join(ind, on=['COD_UPA','NUM_DOM','NUM_UC'], how='left').join(col, on=['COD_UPA','NUM_DOM','NUM_UC'], how='left').fill_null(0))
    merged = merged.with_columns((pl.col('div_ind') + pl.col('div_col')).alias('total_div_tipo'))
    pdf = merged.select(['COD_UPA','NUM_DOM','NUM_UC','total_div_tipo','renda_total']).to_pandas()
    n_total = len(pdf)
    n_with = int((pdf['total_div_tipo'] > 0).sum())
    pct_with = n_with / n_total * 100 if n_total>0 else np.nan
    mean_abs_all = pdf['total_div_tipo'].mean()
    mean_rel_all = (pdf['total_div_tipo'] / pdf['renda_total']).replace([np.inf, -np.inf], np.nan).mean()
    pdf_with = pdf[pdf['total_div_tipo'] > 0]
    if len(pdf_with) > 0:
        mean_abs_with = pdf_with['total_div_tipo'].mean()
        mean_rel_with = (pdf_with['total_div_tipo'] / pdf_with['renda_total']).replace([np.inf, -np.inf], np.nan).mean()
    else:
        mean_abs_with = np.nan
        mean_rel_with = np.nan
    results.append({
        'grupo': name,
        'n_total': n_total,
        'n_with': n_with,
        'pct_with': pct_with,
        'mean_abs_all': mean_abs_all,
        'mean_rel_all': mean_rel_all,
        'mean_abs_with': mean_abs_with,
        'mean_rel_with': mean_rel_with
    })
res_df = pd.DataFrame(results).set_index('grupo')

In [ ]:
# 7.3  Gráficos descritivos (Plotly interativo, por tipo de dívida)  
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import polars as pl
import pandas as pd

# Ajuste opcional: mínimo de UCs para exibir grupo (0 = sem filtro)
min_n = 0  # coloque 30 para filtrar grupos com N<30

# Função auxiliar para montar os 2x3 painéis (mesmo layout usado anteriormente)
def generate_panels(df, title_suffix=''):
    if df.empty:
        print(f"Dados vazios para {title_suffix}, pulando.")
        return

    tab_desc = df.groupby('escolaridade_cat', observed=True).agg(
        n_ucs=('total_divida', 'count'),
        divida_media=('total_divida', 'mean'),
        divida_mediana=('total_divida', 'median'),
        razao_media=('razao_divida_renda', 'mean'),
        razao_mediana=('razao_divida_renda', 'median'),
        renda_media=('renda_total', 'mean'),
        idade_media=('idade_media', 'mean'),
        escolaridade_media=('escolaridade_media', 'mean'),
    ).round(4).reset_index()

    pct_divida = (df.groupby('escolaridade_cat', observed=True)['tem_divida'].mean() * 100).reset_index(name='pct_divida')

    fig = make_subplots(rows=2, cols=3, subplot_titles=(
        'Razão Dívida/Renda Média por Escolaridade',
        '% Famílias com Alguma Dívida por Escolaridade',
        'Renda Média por Escolaridade',
        'Dívida Absoluta Média (referência)',
        'Distribuição Razão Dívida/Renda (UCs c/ dívida, até p95)',
        'Dispersão: Escolaridade vs. Razão Dívida/Renda'
    ))

    # Painel 1: razão média
    fig.add_trace(go.Bar(x=tab_desc['escolaridade_cat'].astype(str), y=tab_desc['razao_media'], marker_color='#636EFA', name='Razão média'), row=1, col=1)

    # Painel 2: % famílias com dívida
    fig.add_trace(go.Bar(x=pct_divida['escolaridade_cat'].astype(str), y=pct_divida['pct_divida'], marker_color='#EF553B', name='% com dívida'), row=1, col=2)

    # Painel 3: renda média
    fig.add_trace(go.Bar(x=tab_desc['escolaridade_cat'].astype(str), y=tab_desc['renda_media'], marker_color='#00CC96', name='Renda média'), row=1, col=3)

    # Painel 4: dívida absoluta média
    fig.add_trace(go.Bar(x=tab_desc['escolaridade_cat'].astype(str), y=tab_desc['divida_media'], marker_color='#FF7F0E', name='Dívida média'), row=2, col=1)

    # Painel 5: boxplot razão dívida/renda (filtra e limita ao pct95)
    df_com_divida = df[df['razao_divida_renda'] > 0].copy()
    if not df_com_divida.empty:
        limite = df_com_divida['razao_divida_renda'].quantile(0.95)
        df_plot = df_com_divida[df_com_divida['razao_divida_renda'] <= limite]
        for cat in df_plot['escolaridade_cat'].cat.categories:
            subset = df_plot[df_plot['escolaridade_cat'] == cat]
            fig.add_trace(go.Box(y=subset['razao_divida_renda'], name=str(cat), marker_color='#636EFA'), row=2, col=2)

    # Painel 6: dispersão — sample para performance
    positive = df[df['razao_divida_renda'] > 0]
    n_sample = min(5000, len(positive))
    if n_sample > 0:
        sample = positive.sample(n=n_sample, random_state=42)
        fig.add_trace(go.Scatter(x=sample['escolaridade_media'], y=sample['razao_divida_renda'].clip(upper=sample['razao_divida_renda'].quantile(0.95)), mode='markers', marker=dict(size=5, opacity=0.4, color='#5A5AFC')), row=2, col=3)

    fig.update_layout(height=800, width=1200, title_text=f'Escolaridade e Endividamento Familiar — {title_suffix}', showlegend=False)
    fig.update_xaxes(tickangle=30)
    fig.show()


# Fonte de dados principal: usa `df_final` (agregado UC) quando disponível
if 'df_final' in globals():
    df_all = df_final.to_pandas()
elif 'df_pandas' in globals():
    df_all = df_pandas.copy()
else:
    df_all = pd.DataFrame()

if not df_all.empty:
    # garante a categoria de escolaridade (mesma escolha que células anteriores)
    df_all['escolaridade_cat'] = pd.cut(
        df_all['escolaridade_media'],
        bins=[-0.01, 4, 8, 11, 100],
        labels=['Nenhuma/Fundamental I (0-4)', 'Fundamental II (4-8)', 'Médio (8-11)', 'Superior (11+)']
    )
    # gera painéis para todas as dívidas somadas (comportamento anterior)
    generate_panels(df_all, 'Todas as dívidas (soma)')

# Definição dos grupos de dívida — usa `debt_groups` se disponível
if 'debt_groups' in globals() and isinstance(debt_groups, dict):
    group_defs = debt_groups
else:
    group_defs = {
        'todas_dividas': codigos_divida if 'codigos_divida' in globals() else [],
        'dividas_amortizacao': list(dividas_amortizacao.keys()) if 'dividas_amortizacao' in globals() else [],
        'dividas_custo': list(dividas_custo.keys()) if 'dividas_custo' in globals() else [],
        'dividas_inadimplencia': list(dividas_inadimplencia.keys()) if 'dividas_inadimplencia' in globals() else [],
        'dividas_atraso': list(dividas_atraso.keys()) if 'dividas_atraso' in globals() else [],
    }

# chaves de agregação (fallback)
key_cols = globals().get('key_cols', ['COD_UPA','NUM_DOM','NUM_UC'])

# Loop por grupo e geração de painéis
for grp_name, codes in group_defs.items():
    # Normaliza lista de códigos
    try:
        codes_list = list(codes) if codes is not None else []
    except Exception:
        codes_list = []

    # Se for 'todas_dividas', reaproveita df_all se disponível
    if grp_name in ('todas_dividas', 'total', 'all') and not df_all.empty:
        pdf = df_all.copy()
        pdf['razao_divida_renda'] = pdf['total_divida'] / pdf['renda_total'].replace({0:np.nan})
        pdf['tem_divida'] = (pdf['total_divida'] > 0).astype(int)
        pdf['escolaridade_cat'] = pd.cut(pdf['escolaridade_media'], bins=[-0.01,4,8,11,100], labels=['Nenhuma/Fundamental I (0-4)','Fundamental II (4-8)','Médio (8-11)','Superior (11+)'])
        print(f"Gerando painéis para: {grp_name} — UCs: {len(pdf)} — UCs c/ dívida: {int((pdf['total_divida']>0).sum())}")
        if len(pdf) >= min_n:
            generate_panels(pdf, grp_name)
        else:
            print(f"Pulando {grp_name} (N={len(pdf)} < min_n={min_n})")
        continue

    # Caso contrário, recomputa agregação (polars) por códigos
    try:
        if not codes_list:
            # cria DataFrames vazios com esquema que inclui as colunas de soma
            empty_schema_ind = {k: pl.Utf8 for k in key_cols}
            empty_schema_ind['div_ind'] = pl.Float64
            ind = pl.DataFrame(schema=empty_schema_ind)

            empty_schema_col = {k: pl.Utf8 for k in key_cols}
            empty_schema_col['div_col'] = pl.Float64
            col = pl.DataFrame(schema=empty_schema_col)
        else:
            ind = despesa_ind.filter(pl.col('V9001').is_in(codes_list)).group_by(key_cols).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_ind'))
            col = despesa_col.filter(pl.col('V9001').is_in(codes_list)).group_by(key_cols).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_col'))
            # garante colunas de soma mesmo se a agregação não retornar a coluna
            if 'div_ind' not in ind.columns:
                ind = ind.with_columns(pl.lit(0.0).alias('div_ind'))
            if 'div_col' not in col.columns:
                col = col.with_columns(pl.lit(0.0).alias('div_col'))
    except Exception as e:
        print(f"Erro ao agregar despesas para {grp_name}: {e}")
        empty_schema_ind = {k: pl.Utf8 for k in key_cols}
        empty_schema_ind['div_ind'] = pl.Float64
        ind = pl.DataFrame(schema=empty_schema_ind)
        empty_schema_col = {k: pl.Utf8 for k in key_cols}
        empty_schema_col['div_col'] = pl.Float64
        col = pl.DataFrame(schema=empty_schema_col)

    # Junta com morador_agg (polars) e transforma para pandas
    try:
        merged = morador_agg.join(ind, on=key_cols, how='left').join(col, on=key_cols, how='left').fill_null(0)
        merged = merged.with_columns((pl.col('div_ind').fill_null(0) + pl.col('div_col').fill_null(0)).alias('total_divida'))
        # seleciona colunas mínimas para compatibilidade
        sel_cols = [c for c in ['COD_UPA','NUM_DOM','NUM_UC','total_divida','renda_total','escolaridade_media','idade_media'] if c in merged.columns]
        pdf = merged.select(sel_cols).to_pandas()
    except Exception as e:
        print(f"Erro ao juntar morador_agg para {grp_name}: {e}")
        pdf = pd.DataFrame()

    if pdf.empty:
        print(f"Nenhuma UC encontrada para {grp_name}, pulando.")
        continue

    # cria variáveis derivadas
    with np.errstate(divide='ignore', invalid='ignore'):
        pdf['razao_divida_renda'] = pdf['total_divida'] / pdf['renda_total'].replace({0:np.nan})
    pdf['tem_divida'] = (pdf['total_divida'] > 0).astype(int)
    pdf['escolaridade_cat'] = pd.cut(pdf['escolaridade_media'].fillna(0), bins=[-0.01,4,8,11,100], labels=['Nenhuma/Fundamental I (0-4)','Fundamental II (4-8)','Médio (8-11)','Superior (11+)'])

    print(f"Gerando painéis para: {grp_name} — UCs: {len(pdf)} — UCs c/ dívida: {int((pdf['total_divida']>0).sum())}")
    if len(pdf) >= min_n:
        generate_panels(pdf, grp_name)
    else:
        print(f"Pulando {grp_name} (N={len(pdf)} < min_n={min_n})")

print('✓ Gráficos interativos por tipo de dívida gerados (Plotly).')


In [ ]:
# 7.4  Análises por tipo de dívida (Plotly: tabela e gráficos resumidos)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Recarrega res_df se necessário (calculado na célula de análises por grupo)
try:
    res_df
except NameError:
    # tenta recriar (caso o notebook seja executado fora de ordem)
    pass

# Se a variável res_df não existir, executa a agregação leve (como fallback)
if 'res_df' not in globals():
    debt_groups = {
        'todas_dividas': codigos_divida,
        'dividas_custo': list(dividas_custo.keys()),
        'dividas_amortizacao': list(dividas_amortizacao.keys()),
        'dividas_inadimplencia': list(dividas_inadimplencia.keys()),
        'dividas_atraso': list(dividas_atraso.keys()),
    }
    results = []
    for name, codes in debt_groups.items():
        ind = despesa_ind.filter(pl.col('V9001').is_in(codes)).group_by(['COD_UPA','NUM_DOM','NUM_UC']).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_ind'))
        if despesa_col.shape[0] > 0:
            col = despesa_col.filter(pl.col('V9001').is_in(codes)).group_by(['COD_UPA','NUM_DOM','NUM_UC']).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_col'))
        else:
            col = pl.DataFrame({'COD_UPA': pl.Series([], dtype=pl.Utf8), 'NUM_DOM': pl.Series([], dtype=pl.Utf8), 'NUM_UC': pl.Series([], dtype=pl.Utf8), 'div_col': pl.Series([], dtype=pl.Float64)})
        merged = (morador_agg.join(ind, on=['COD_UPA','NUM_DOM','NUM_UC'], how='left').join(col, on=['COD_UPA','NUM_DOM','NUM_UC'], how='left').fill_null(0))
        merged = merged.with_columns((pl.col('div_ind') + pl.col('div_col')).alias('total_div_tipo'))
        pdf = merged.select(['COD_UPA','NUM_DOM','NUM_UC','total_div_tipo','renda_total']).to_pandas()
        n_total = len(pdf)
        n_with = int((pdf['total_div_tipo'] > 0).sum())
        pct_with = n_with / n_total * 100 if n_total>0 else np.nan
        mean_abs_all = pdf['total_div_tipo'].mean()
        mean_rel_all = (pdf['total_div_tipo'] / pdf['renda_total']).replace([np.inf, -np.inf], np.nan).mean()
        pdf_with = pdf[pdf['total_div_tipo'] > 0]
        if len(pdf_with) > 0:
            mean_abs_with = pdf_with['total_div_tipo'].mean()
            mean_rel_with = (pdf_with['total_div_tipo'] / pdf_with['renda_total']).replace([np.inf, -np.inf], np.nan).mean()
        else:
            mean_abs_with = np.nan
            mean_rel_with = np.nan
        results.append({
            'grupo': name,
            'n_total': n_total,
            'n_with': n_with,
            'pct_with': pct_with,
            'mean_abs_all': mean_abs_all,
            'mean_rel_all': mean_rel_all,
            'mean_abs_with': mean_abs_with,
            'mean_rel_with': mean_rel_with
        })
    res_df = pd.DataFrame(results).set_index('grupo')

# Prepara tabela legível: renomeia colunas explicativas e arredonda para 2 casas decimais
display_df = res_df.reset_index().rename(columns={
    'grupo': 'Grupo de Dívida',
    'n_total': 'UCs (total)',
    'n_with': 'UCs com gasto',
    'pct_with': '% UCs com gasto',
    'mean_abs_all': 'Média dívida (incluindo todos) R$',
    'mean_rel_all': 'Média dívida/renda (incluindo todos)',
    'mean_abs_with': 'Média dívida (UCs c/ gasto) R$',
    'mean_rel_with': 'Média dívida/renda (UCs c/ gasto)'
})

# Arredonda números a 2 casas decimais para exibição
num_cols = ['UCs (total)', 'UCs com gasto', '% UCs com gasto', 'Média dívida (incluindo todos) R$', 'Média dívida/renda (incluindo todos)', 'Média dívida (UCs c/ gasto) R$', 'Média dívida/renda (UCs c/ gasto)']
for c in num_cols:
    if c in display_df.columns:
        display_df[c] = display_df[c].apply(lambda x: (round(x,2) if pd.notna(x) else x))

# Exibe tabela interativa com Plotly com colunas de largura ajustada
col_vals = [display_df[c].tolist() for c in display_df.columns]
col_names = list(display_df.columns)

# Define larguras (em pixels) apropriadas para cada coluna
col_widths = [220, 100, 110, 120, 180, 180, 180, 180][:len(col_names)]

table_fig = go.Figure(data=[go.Table(
    header=dict(values=col_names, fill_color='lightgrey', align='left', font=dict(size=11)),
    cells=dict(values=col_vals, align='left', font=dict(size=10)),
    columnwidth=col_widths
)])

table_fig.update_layout(title='Resumo: estatísticas por grupo de dívida (valores arredondados)', height=420, width=1200)
table_fig.show()

# Gráficos resumidos: % UCs com gasto e média dívida/renda (incluindo todos)
color_pct = '#636EFA'   # azul Plotly padrão
color_mean = '#EF553B'  # laranja/vermelho Plotly padrão

# Garante que usamos valores numéricos não-arredondados para cálculo, mas mostramos 2 casas nos labels
plot_df = res_df.copy()
plot_df_display = plot_df.copy()
plot_df_display['pct_with'] = plot_df_display['pct_with'].round(2)
plot_df_display['mean_rel_all'] = plot_df_display['mean_rel_all'].round(4)  # mostrar 4 decimais se for proporção

fig = make_subplots(rows=1, cols=2, subplot_titles=['% UCs com gasto por grupo', 'Média dívida/renda (incluindo todos)'])
fig.add_trace(go.Bar(x=plot_df.index.astype(str), y=plot_df['pct_with'], marker_color=color_pct, name='% UCs com gasto',
                     text=plot_df_display['pct_with'].astype(str) + '%', textposition='auto', hovertemplate='%{x}<br>%{y:.2f}%'), row=1, col=1)
fig.add_trace(go.Bar(x=plot_df.index.astype(str), y=plot_df['mean_rel_all'], marker_color=color_mean, name='Média dívida/renda',
                     text=plot_df_display['mean_rel_all'].astype(str), textposition='auto', hovertemplate='%{x}<br>%{y:.4f}'), row=1, col=2)

fig.update_layout(height=460, width=1200, showlegend=False)
fig.update_xaxes(tickangle=30)
fig.show()

# Salva novamente para garantia de reprodutibilidade
res_df.to_csv('analise_dividas_por_grupo_plotly.csv')
print('✓ Tabela e gráficos Plotly gerados; CSV salvo: analise_dividas_por_grupo_plotly.csv')


## 8. Modelos de Regressão

### Justificativa metodológica

A variável `total_divida` (e a `razao_divida_renda`) tem uma distribuição
**extremamente assimétrica**: a maioria das UCs não tem dívida alguma
(total_divida = 0), e as que têm apresentam valores muito dispersos.
Isso viola os pressupostos do OLS clássico.

Um estatístico sênior usaria uma **abordagem em duas partes** (*two-part model*
ou *hurdle model*), que separa o fenômeno em duas perguntas distintas:

1. **Parte 1 — Regressão Logística**: *Qual é a probabilidade de uma UC
   TER alguma dívida?*
   - Variável dependente: `tem_divida` (0/1)
   - Captura o acesso ao crédito

2. **Parte 2 — OLS log-linear**: *Dado que a UC TEM dívida, qual é o
   montante (em log)?*
   - Variável dependente: `log(total_divida)` — normaliza a distribuição
   - Captura o volume de endividamento

Também rodamos um OLS na **razão dívida/renda** (com erro padrão robusto HC3)
como análise complementar sobre o comprometimento relativo da renda.

### Variáveis de controle
- `renda_total`: controla capacidade de pagamento
- `idade_media`: controla ciclo de vida
- `nivel_instrucao_modal`: complementa `escolaridade_media` (nível categórico)
- `n_membros`: tamanho da UC
- `sexo_chefe`: dummies de gênero do responsável
- `UF`: efeitos fixos de estado (controla desigualdades regionais)


In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan

# Prepara o dataframe de modelagem
df_model = df_pandas.copy()

# Log da dívida (apenas para UCs com dívida > 0 — usado no segundo estágio)
df_model['log_divida'] = np.where(
    df_model['total_divida'] > 0,
    np.log(df_model['total_divida']),
    np.nan
)

# Log da renda (para modelar elasticidade)
df_model['log_renda'] = np.log(df_model['renda_total'])

# Variáveis dummy
df_model['chefe_mulher'] = (df_model['sexo_chefe'] == '2').astype(int)
df_model['UF'] = df_model['UF'].astype(str)  # garante tratamento como categórica

# ──────────────────────────────────────────────────────────────────────────────
# PARTE 1: Regressão Logística
# Pergunta: A escolaridade aumenta a PROBABILIDADE de ter dívida?
# ──────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("MODELO 1 — REGRESSÃO LOGÍSTICA: P(TER DÍVIDA)")
print("Variável dependente: tem_divida (0/1)")
print("=" * 70)

formula_logit = (
    'tem_divida ~ escolaridade_media + C(nivel_instrucao_modal) '
    '+ log_renda + idade_media + n_membros + chefe_mulher + C(UF)'
)

modelo_logit = smf.logit(formula_logit, data=df_model).fit(maxiter=200, disp=False)
print(modelo_logit.summary())

# Odds-ratio para escolaridade
or_esc = np.exp(modelo_logit.params['escolaridade_media'])
ci_or  = np.exp(modelo_logit.conf_int().loc['escolaridade_media'])
print(f"\nOdds-ratio para escolaridade_media: {or_esc:.4f}  IC 95%: [{ci_or[0]:.4f}, {ci_or[1]:.4f}]")
print("  → Se OR > 1: cada ano adicional de estudo aumenta a chance de ter dívida.")
print("  → Se OR < 1: cada ano adicional de estudo reduz a chance de ter dívida.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# PARTE 2: OLS log-linear (somente UCs com dívida > 0)
# Pergunta: Dado que a UC tem dívida, a escolaridade aumenta o VOLUME?
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("MODELO 2 — OLS LOG-LINEAR: log(DÍVIDA) | dívida > 0")
print("Variável dependente: log(total_divida)  (apenas UCs com dívida > 0)")
print("=" * 70)

df_com_divida = df_model.dropna(subset=['log_divida']).copy()

formula_ols_log = (
    'log_divida ~ escolaridade_media + C(nivel_instrucao_modal) '
    '+ log_renda + idade_media + n_membros + chefe_mulher + C(UF)'
)

modelo_ols_log = smf.ols(formula_ols_log, data=df_com_divida).fit(
    cov_type='HC3'  # erros padrão robustos à heterocedasticidade
)
print(modelo_ols_log.summary())

beta_esc = modelo_ols_log.params['escolaridade_media']
pval_esc = modelo_ols_log.pvalues['escolaridade_media']
ci_esc   = modelo_ols_log.conf_int().loc['escolaridade_media']

print(f"\nCoeficiente de escolaridade_media: {beta_esc:.4f}")
print(f"  → Interpretação (log-linear): cada ano adicional de estudo está associado")
print(f"    a uma variação de {beta_esc*100:.1f}% no montante de dívida,")
print(f"    ceteris paribus (controlando renda, idade, UF, etc.).")
print(f"  P-valor: {pval_esc:.4f}  |  IC 95%: [{ci_esc[0]:.4f}, {ci_esc[1]:.4f}]")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# COMPLEMENTAR: OLS na razão dívida/renda (todas as UCs)
# Pergunta: A escolaridade altera o COMPROMETIMENTO relativo da renda?
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("MODELO 3 — OLS: RAZÃO DÍVIDA/RENDA (todas as UCs, erros robustos)")
print("Variável dependente: razao_divida_renda")
print("=" * 70)

# Winsoriza a razão no percentil 99 para reduzir influência de outliers extremos
p99 = df_model['razao_divida_renda'].quantile(0.99)
df_model['razao_divida_renda_w'] = df_model['razao_divida_renda'].clip(upper=p99)

formula_razao = (
    'razao_divida_renda_w ~ escolaridade_media + C(nivel_instrucao_modal) '
    '+ idade_media + n_membros + chefe_mulher + C(UF)'
    # renda NÃO entra como controle aqui — ela já está no denominador
)

modelo_razao = smf.ols(formula_razao, data=df_model).fit(cov_type='HC3')
print(modelo_razao.summary())

beta_r = modelo_razao.params['escolaridade_media']
pval_r = modelo_razao.pvalues['escolaridade_media']

print(f"\nCoeficiente de escolaridade_media: {beta_r:.6f}")
print(f"  → Cada ano adicional de estudo altera a razão dívida/renda em {beta_r:.6f} pontos.")
print(f"  P-valor: {pval_r:.4f}")
print()
if pval_r < 0.05:
    direcao = "AUMENTA" if beta_r > 0 else "REDUZ"
    print(f"  ✅ Resultado significante: maior escolaridade {direcao} o comprometimento")
    print(f"     relativo da renda com dívida (controlando por idade, UF, etc.).")
else:
    print("  ⚠️  Resultado NÃO significante para razão dívida/renda.")
    print("     Isso indicaria que, apesar de a dívida absoluta ser maior,")
    print("     o comprometimento relativo da renda pode ser semelhante entre níveis de educação.")

In [ ]:
# 8.1  Modelos por grupo de dívida — análise rigorosa e reprodutível
# Para cada grupo (todas, custo, amortização, inadimplência, atraso) executamos:
#  - Logit: P(ter gasto no grupo) — variável dependente binária
#  - OLS log-linear: log(gasto) | gasto>0 — volume condicional
#  - OLS razão dívida/renda: compromisso relativo (winsorizado)
# Resultados são sumarizados em um CSV para tabelas e apêndice.
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Garante df_model disponível (se célula anterior não foi executada)
try:
    df_model
except NameError:
    try:
        df_model = df_pandas.copy()
    except Exception:
        df_model = df_final.to_pandas()

# Reúne os grupos (usa dicionários já definidos no notebook)
debt_groups = {
    'todas_dividas': codigos_divida,
    'dividas_custo': list(dividas_custo.keys()),
    'dividas_amortizacao': list(dividas_amortizacao.keys()),
    'dividas_inadimplencia': list(dividas_inadimplencia.keys()),
    'dividas_atraso': list(dividas_atraso.keys()),
}

model_summaries = []
for name, codes in debt_groups.items():
    # Agrega gastos por UC para o grupo (individual + coletiva) usando Polars
    ind_g = despesa_ind.filter(pl.col('V9001').is_in(codes)).group_by(['COD_UPA','NUM_DOM','NUM_UC']).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_ind'))
    if despesa_col.shape[0] > 0:
        col_g = despesa_col.filter(pl.col('V9001').is_in(codes)).group_by(['COD_UPA','NUM_DOM','NUM_UC']).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_col'))
    else:
        col_g = pl.DataFrame({
            'COD_UPA': pl.Series([], dtype=pl.Utf8),
            'NUM_DOM': pl.Series([], dtype=pl.Utf8),
            'NUM_UC': pl.Series([], dtype=pl.Utf8),
            'div_col': pl.Series([], dtype=pl.Float64),
        })

    merged = (
        morador_agg
        .join(ind_g, on=['COD_UPA','NUM_DOM','NUM_UC'], how='left')
        .join(col_g, on=['COD_UPA','NUM_DOM','NUM_UC'], how='left')
        .fill_null(0)
    )
    merged = merged.with_columns((pl.col('div_ind') + pl.col('div_col')).alias('total_div_tipo'))

    pdf = merged.select(['COD_UPA','NUM_DOM','NUM_UC','total_div_tipo','renda_total','escolaridade_media','idade_media','n_membros','UF']).to_pandas()

    # junta metadados (sexo_chefe, nivel_instrucao_modal) para manter consistência
    try:
        meta = merged.select(['COD_UPA','NUM_DOM','NUM_UC','sexo_chefe','nivel_instrucao_modal']).to_pandas()
        pdf = pdf.merge(meta, on=['COD_UPA','NUM_DOM','NUM_UC'], how='left')
    except Exception:
        pdf['sexo_chefe'] = np.nan
        pdf['nivel_instrucao_modal'] = np.nan

    pdf['renda_total'] = pdf['renda_total'].astype(float)

    # Cria variáveis para modelagem
    pdf['tem_divida_grp'] = (pdf['total_div_tipo'] > 0).astype(int)
    pdf['debt_ratio_grp'] = (pdf['total_div_tipo'] / pdf['renda_total']).replace([np.inf, -np.inf], np.nan)
    pdf['log_debt_grp'] = np.where(pdf['total_div_tipo'] > 0, np.log(pdf['total_div_tipo']), np.nan)
    pdf['chefe_mulher'] = (pdf['sexo_chefe'] == '2').astype(int)
    pdf['nivel_instrucao_modal'] = pdf['nivel_instrucao_modal'].astype('category')

    n_total = len(pdf)
    n_with = int(pdf['tem_divida_grp'].sum())

    # Formula de controles — igual aos modelos principais (manter comparabilidade)
    controls = ' + C(nivel_instrucao_modal) + log_renda + idade_media + n_membros + chefe_mulher + C(UF)'

    # Prepara df para modelos: junta com df_model se disponível para variáveis auxiliares
    try:
        base = df_model.merge(pdf[['COD_UPA','NUM_DOM','NUM_UC','total_div_tipo','tem_divida_grp','debt_ratio_grp','log_debt_grp']], on=['COD_UPA','NUM_DOM','NUM_UC'], how='left')
    except Exception:
        base = pdf.copy()

    # Logit — P(ter gasto no grupo)
    logit_res = {'success': False}
    try:
        formula = 'tem_divida_grp ~ escolaridade_media' + controls
        mod_logit = smf.logit(formula, data=base).fit(disp=False, maxiter=200)
        coef = mod_logit.params.get('escolaridade_media', np.nan)
        pval = mod_logit.pvalues.get('escolaridade_media', np.nan)
        orr = np.exp(coef) if not np.isnan(coef) else np.nan
        logit_res = {'coef': coef, 'pval': pval, 'or': orr, 'nobs': int(mod_logit.nobs), 'success': True}
    except Exception as e:
        logit_res = {'error': str(e), 'success': False}

    # OLS log-linear (condicional)
    ols_log_res = {'success': False}
    try:
        df_pos = base.dropna(subset=['log_debt_grp']).copy()
        if len(df_pos) > 0:
            formula_l = 'log_debt_grp ~ escolaridade_media' + controls
            mod_l = smf.ols(formula_l, data=df_pos).fit(cov_type='HC3')
            ols_log_res = {'coef': mod_l.params.get('escolaridade_media', np.nan), 'pval': mod_l.pvalues.get('escolaridade_media', np.nan), 'nobs': int(mod_l.nobs), 'success': True}
    except Exception as e:
        ols_log_res = {'error': str(e), 'success': False}

    # OLS razão dívida/renda (winsorizada 99%)
    ols_ratio_res = {'success': False}
    try:
        base['debt_ratio_w'] = base['debt_ratio_grp'].clip(upper=base['debt_ratio_grp'].quantile(0.99))
        formula_r = 'debt_ratio_w ~ escolaridade_media + C(nivel_instrucao_modal) + idade_media + n_membros + chefe_mulher + C(UF)'
        mod_r = smf.ols(formula_r, data=base).fit(cov_type='HC3')
        ols_ratio_res = {'coef': mod_r.params.get('escolaridade_media', np.nan), 'pval': mod_r.pvalues.get('escolaridade_media', np.nan), 'nobs': int(mod_r.nobs), 'success': True}
    except Exception as e:
        ols_ratio_res = {'error': str(e), 'success': False}

    # Agrupa resultados
    model_summaries.append({
        'grupo': name,
        'n_total': n_total,
        'n_with': n_with,
        'pct_with': (n_with / n_total * 100) if n_total>0 else np.nan,
        'logit': logit_res,
        'ols_log': ols_log_res,
        'ols_ratio': ols_ratio_res,
    })

# Constrói DataFrame legível para exportação
rows = []
for s in model_summaries:
    rows.append({
        'grupo': s['grupo'],
        'n_total': s['n_total'],
        'n_with': s['n_with'],
        'pct_with': s['pct_with'],
        'logit_coef': s['logit'].get('coef') if isinstance(s['logit'], dict) else np.nan,
        'logit_pval': s['logit'].get('pval') if isinstance(s['logit'], dict) else np.nan,
        'logit_or': s['logit'].get('or') if isinstance(s['logit'], dict) else np.nan,
        'ols_log_coef': s['ols_log'].get('coef') if isinstance(s['ols_log'], dict) else np.nan,
        'ols_log_pval': s['ols_log'].get('pval') if isinstance(s['ols_log'], dict) else np.nan,
        'ols_log_n': s['ols_log'].get('nobs') if isinstance(s['ols_log'], dict) else np.nan,
        'ols_ratio_coef': s['ols_ratio'].get('coef') if isinstance(s['ols_ratio'], dict) else np.nan,
        'ols_ratio_pval': s['ols_ratio'].get('pval') if isinstance(s['ols_ratio'], dict) else np.nan,
        'ols_ratio_n': s['ols_ratio'].get('nobs') if isinstance(s['ols_ratio'], dict) else np.nan,
    })
out_df = pd.DataFrame(rows).set_index('grupo')
out_df.to_csv('modelos_por_grupo.csv')
print('\n✅ Modelos por grupo executados e salvos em modelos_por_grupo.csv')
print(out_df)


In [ ]:
# 9 Diagnósticos e ANOVA para as análises por grupo de dívida
# Produz diagnósticos de resíduos (Jarque-Bera, Breusch-Pagan) para os OLS
# e testa diferenças entre níveis de escolaridade (ANOVA e Kruskal-Wallis)
import numpy as np
import pandas as pd
from scipy.stats import jarque_bera, f_oneway, kruskal
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Garante que debt_groups exista (definido na Seção 8.1)
try:
    debt_groups
except NameError:
    debt_groups = {
        'todas_dividas': codigos_divida,
        'dividas_custo': list(dividas_custo.keys()),
        'dividas_amortizacao': list(dividas_amortizacao.keys()),
        'dividas_inadimplencia': list(dividas_inadimplencia.keys()),
        'dividas_atraso': list(dividas_atraso.keys()),
    }

diag_rows = []
anova_rows = []

# Mesmas categorias de escolaridade usadas antes
escol_bins = [-0.01, 4, 8, 11, 16]
escol_labels = ['Nenhuma/Fundamental I (0-4)', 'Fundamental II (4-8)', 'Médio (8-11)', 'Superior (11+)']

for name, codes in debt_groups.items():
    # Recria base agregada por UC (mesma lógica das células anteriores)
    ind_g = despesa_ind.filter(pl.col('V9001').is_in(codes)).group_by(['COD_UPA','NUM_DOM','NUM_UC']).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_ind'))
    if despesa_col.shape[0] > 0:
        col_g = despesa_col.filter(pl.col('V9001').is_in(codes)).group_by(['COD_UPA','NUM_DOM','NUM_UC']).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_col'))
    else:
        col_g = pl.DataFrame({'COD_UPA': pl.Series([], dtype=pl.Utf8), 'NUM_DOM': pl.Series([], dtype=pl.Utf8), 'NUM_UC': pl.Series([], dtype=pl.Utf8), 'div_col': pl.Series([], dtype=pl.Float64)})

    merged = (
        morador_agg
        .join(ind_g, on=['COD_UPA','NUM_DOM','NUM_UC'], how='left')
        .join(col_g, on=['COD_UPA','NUM_DOM','NUM_UC'], how='left')
        .fill_null(0)
    )
    merged = merged.with_columns((pl.col('div_ind') + pl.col('div_col')).alias('total_div_tipo'))
    pdf = merged.select(['COD_UPA','NUM_DOM','NUM_UC','total_div_tipo','renda_total','escolaridade_media','idade_media','n_membros','UF']).to_pandas()

    # Variáveis para testes
    pdf['tem_divida_grp'] = (pdf['total_div_tipo'] > 0).astype(int)
    pdf['debt_ratio_grp'] = (pdf['total_div_tipo'] / pdf['renda_total']).replace([np.inf, -np.inf], np.nan)
    pdf['log_debt_grp'] = np.where(pdf['total_div_tipo'] > 0, np.log(pdf['total_div_tipo']), np.nan)
    pdf['escolaridade_cat'] = pd.cut(pdf['escolaridade_media'], bins=escol_bins, labels=escol_labels)

    # ---- Diagnósticos dos modelos OLS (condicional e razão) ----
    # OLS log-linear (condicional)
    try:
        df_pos = pdf.dropna(subset=['log_debt_grp']).copy()
        if len(df_pos) > 0:
            mod_l = smf.ols('log_debt_grp ~ escolaridade_media + C(escolaridade_cat) + idade_media + n_membros + C(UF)', data=df_pos).fit(cov_type='HC3')
            resid_l = mod_l.resid
            jb_stat, jb_p = jarque_bera(resid_l)
            try:
                bp_lm, bp_p, _, _ = het_breuschpagan(resid_l, mod_l.model.exog)
            except Exception as e:
                bp_lm, bp_p = np.nan, np.nan
        else:
            mod_l = None
            jb_stat, jb_p, bp_lm, bp_p = [np.nan]*4
    except Exception as e:
        mod_l = None
        jb_stat, jb_p, bp_lm, bp_p = [np.nan]*4

    # OLS razão dívida/renda (todas as UCs, winsoriza 99%)
    try:
        pdf['debt_ratio_w'] = pdf['debt_ratio_grp'].clip(upper=pdf['debt_ratio_grp'].quantile(0.99))
        mod_r = smf.ols('debt_ratio_w ~ escolaridade_media + C(escolaridade_cat) + idade_media + n_membros + C(UF)', data=pdf).fit(cov_type='HC3')
        resid_r = mod_r.resid
        jb_stat_r, jb_p_r = jarque_bera(resid_r)
        try:
            bp_lm_r, bp_p_r, _, _ = het_breuschpagan(resid_r, mod_r.model.exog)
        except Exception:
            bp_lm_r, bp_p_r = np.nan, np.nan
    except Exception:
        mod_r = None
        jb_stat_r, jb_p_r, bp_lm_r, bp_p_r = [np.nan]*4

    diag_rows.append({
        'grupo': name,
        'n_total': len(pdf),
        'n_with': int(pdf['tem_divida_grp'].sum()),
        'pct_with': (pdf['tem_divida_grp'].mean()*100),
        'jb_log_stat': jb_stat,
        'jb_log_p': jb_p,
        'bp_log_lm': bp_lm,
        'bp_log_p': bp_p,
        'jb_ratio_stat': jb_stat_r,
        'jb_ratio_p': jb_p_r,
        'bp_ratio_lm': bp_lm_r,
        'bp_ratio_p': bp_p_r,
    })

    # ---- ANOVA / Kruskal-Wallis: debt_ratio por escolaridade ----
    try:
        grupos_all = [pdf.loc[pdf['escolaridade_cat']==cat, 'debt_ratio_grp'].dropna().values for cat in escol_labels]
        # ANOVA (note: may be invalid due to heterocedasticidade/zeros) -- report p-value
        try:
            f_stat, p_anova = f_oneway(*[g for g in grupos_all if len(g)>0])
        except Exception:
            f_stat, p_anova = np.nan, np.nan
        # Kruskal-Wallis (mais robusto)
        try:
            h_stat, p_kw = kruskal(*[g for g in grupos_all if len(g)>0])
        except Exception:
            h_stat, p_kw = np.nan, np.nan

        # Mesma análise apenas para UCs com dívida
        pdf_with = pdf[pdf['tem_divida_grp']>0]
        grupos_with = [pdf_with.loc[pdf_with['escolaridade_cat']==cat, 'debt_ratio_grp'].dropna().values for cat in escol_labels]
        try:
            f_stat_w, p_anova_w = f_oneway(*[g for g in grupos_with if len(g)>0])
        except Exception:
            f_stat_w, p_anova_w = np.nan, np.nan
        try:
            h_stat_w, p_kw_w = kruskal(*[g for g in grupos_with if len(g)>0])
        except Exception:
            h_stat_w, p_kw_w = np.nan, np.nan

    except Exception:
        f_stat, p_anova, h_stat, p_kw = [np.nan]*4
        f_stat_w, p_anova_w, h_stat_w, p_kw_w = [np.nan]*4

    anova_rows.append({
        'grupo': name,
        'n_total': len(pdf),
        'n_with': int(pdf['tem_divida_grp'].sum()),
        'ANOVA_f': f_stat,
        'ANOVA_p': p_anova,
        'Kruskal_h': h_stat,
        'Kruskal_p': p_kw,
        'ANOVA_f_with': f_stat_w,
        'ANOVA_p_with': p_anova_w,
        'Kruskal_h_with': h_stat_w,
        'Kruskal_p_with': p_kw_w,
    })

# Salva e exibe resultados de forma legível
diag_df = pd.DataFrame(diag_rows).set_index('grupo')
anova_df = pd.DataFrame(anova_rows).set_index('grupo')

diag_df.to_csv('diagnosticos_por_grupo.csv')
anova_df.to_csv('anova_por_grupo.csv')

pd.options.display.float_format = '{:,.4f}'.format
print('\n=== Diagnósticos (resumo) ===')
print(diag_df)
print('\n=== ANOVA / Kruskal (resumo) ===')
print(anova_df)

print('\nArquivos salvos: diagnosticos_por_grupo.csv, anova_por_grupo.csv')


## 9. Diagnósticos dos Modelos

In [ ]:
from scipy.stats import jarque_bera, shapiro

print("=" * 70)
print("DIAGNÓSTICOS — MODELO 2 (OLS log-linear)")
print("=" * 70)

residuos = modelo_ols_log.resid

# Jarque-Bera (normalidade dos resíduos)
jb_stat, jb_p = jarque_bera(residuos)
print(f"\nJarque-Bera (normalidade):  estatística={jb_stat:.1f}, p={jb_p:.4f}")
print(f"  {'✗ Resíduos não normais' if jb_p < 0.05 else '✓ Resíduos aproximadamente normais'}")
print("  [NOTA] Com N grande (>30.000), JB quase sempre rejeita normalidade.")
print("         O TLC garante que os estimadores OLS são assintoticamente válidos.")
print("         O uso de erros robustos HC3 protege a inferência.")

# Breusch-Pagan (heterocedasticidade)
try:
    bp_lm, bp_p, _, _ = het_breuschpagan(residuos, modelo_ols_log.model.exog)
    print(f"\nBreusch-Pagan (homocedasticidade): LM={bp_lm:.1f}, p={bp_p:.4f}")
    print(f"  {'✗ Heterocedasticidade detectada' if bp_p < 0.05 else '✓ Variância aproximadamente constante'}")
    print("  [NOTA] Heterocedasticidade é esperada em dados de renda/despesa.")
    print("         Erros padrão HC3 já corrigem isso — inferência válida.")
except Exception as e:
    print(f"  (Teste BP não disponível: {e})")

# Gráficos diagnósticos
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Diagnósticos — Modelo 2 (OLS log-linear)', fontsize=12)

# Resíduos vs. valores ajustados
axes[0].scatter(modelo_ols_log.fittedvalues, residuos, alpha=0.2, s=8)
axes[0].axhline(0, color='red', linewidth=1)
axes[0].set_xlabel('Valores ajustados (log dívida)')
axes[0].set_ylabel('Resíduos')
axes[0].set_title('Resíduos vs. Ajustados')
axes[0].grid(True, alpha=0.3)

# QQ-plot dos resíduos
sm.qqplot(residuos, line='s', ax=axes[1], alpha=0.3)
axes[1].set_title('Q-Q Plot dos Resíduos')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Comparação de modelos
print("\n" + "=" * 70)
print("COMPARAÇÃO DE MODELOS")
print("=" * 70)
print(f"{'Métrica':<35} {'Mod.2 OLS-log':<20} {'Mod.3 Razão':<20}")
print("-" * 75)
print(f"{'R-quadrado':<35} {modelo_ols_log.rsquared:<20.4f} {modelo_razao.rsquared:<20.4f}")
print(f"{'R-quad. ajustado':<35} {modelo_ols_log.rsquared_adj:<20.4f} {modelo_razao.rsquared_adj:<20.4f}")
print(f"{'AIC':<35} {modelo_ols_log.aic:<20.1f} {modelo_razao.aic:<20.1f}")
print(f"{'N observações':<35} {int(modelo_ols_log.nobs):<20} {int(modelo_razao.nobs):<20}")
print(f"{'β escolaridade_media':<35} {modelo_ols_log.params['escolaridade_media']:<20.4f} {modelo_razao.params['escolaridade_media']:<20.6f}")
print(f"{'p-valor escolaridade':<35} {modelo_ols_log.pvalues['escolaridade_media']:<20.4f} {modelo_razao.pvalues['escolaridade_media']:<20.4f}")

## Teste ANOVA e sumário final

In [ ]:
from scipy.stats import f_oneway, kruskal

print("=" * 70)
print("TESTES DE DIFERENÇA ENTRE GRUPOS DE ESCOLARIDADE")
print("=" * 70)

grupos_razao = [
    df_pandas[df_pandas['escolaridade_cat'] == cat]['razao_divida_renda'].dropna().values
    for cat in df_pandas['escolaridade_cat'].cat.categories
]

# ANOVA paramétrica
f_stat, p_anova = f_oneway(*grupos_razao)
print(f"\nANOVA (razão dívida/renda): F={f_stat:.2f}, p={p_anova:.2e}")
print(f"  {'✓ Diferenças significantes entre grupos' if p_anova < 0.05 else '✗ Sem diferenças significantes'}")

# Kruskal-Wallis (não-paramétrico — mais robusto para dados assimétricos)
h_stat, p_kw = kruskal(*grupos_razao)
print(f"\nKruskal-Wallis (razão dívida/renda): H={h_stat:.2f}, p={p_kw:.2e}")
print(f"  {'✓ Diferenças significantes entre grupos' if p_kw < 0.05 else '✗ Sem diferenças significantes'}")
print("  [NOTA] Kruskal-Wallis é preferível para dados com muitos zeros e assimetria.")

print("\n" + "=" * 70)
print("SUMÁRIO EXECUTIVO")
print("=" * 70)
print(f"""
Dataset: {len(df_pandas):,} UCs (famílias) da POF 2017-2018
UCs com alguma dívida: {df_pandas['tem_divida'].sum():,} ({df_pandas['tem_divida'].mean()*100:.1f}%)

MODELO 1 — Logístico (P(ter dívida)):
  Odds-ratio escolaridade: {np.exp(modelo_logit.params['escolaridade_media']):.4f}
  p-valor: {modelo_logit.pvalues['escolaridade_media']:.4f}

MODELO 2 — OLS log-linear (log dívida | dívida > 0):
  β escolaridade: {modelo_ols_log.params['escolaridade_media']:.4f}
  → Cada ano adicional de estudo associado a {modelo_ols_log.params['escolaridade_media']*100:.1f}% no montante de dívida
  p-valor: {modelo_ols_log.pvalues['escolaridade_media']:.4f}

MODELO 3 — OLS razão dívida/renda:
  β escolaridade: {modelo_razao.params['escolaridade_media']:.6f}
  p-valor: {modelo_razao.pvalues['escolaridade_media']:.4f}

CONCLUSÃO PRELIMINAR:
  Compare os três modelos para uma narrativa completa:
  - Se o logístico for sig. positivo: escolaridade aumenta ACESSO ao crédito
  - Se o OLS-log for sig. positivo: dado o acesso, escolaridade também aumenta VOLUME
  - Se a razão for sig. positiva: o comprometimento relativo da renda também é maior
  - Se a razão NÃO for sig.: a dívida cresce proporcionalmente à renda (sem sobreendividamento)
""")

In [ ]:
# **Seção: Modelos — relação entre escolaridade (UC) e dívida (medidas: max, median, média)**
# Executa regressões para cada medida de escolaridade (max, median, média), para cada grupo de dívida,
# e para duas amostras: todas as UCs e apenas UCs com dívida (>0). Salva CSV final.

import polars as pl
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

key_cols = ['COD_UPA','NUM_DOM','NUM_UC']

# Grupos de dívida definidos no notebook (inclui extras já definidos)
debt_groups = {
    'todas_dividas': codigos_divida if 'codigos_divida' in globals() else [],
    'dividas_amortizacao': list(dividas_amortizacao.keys()) if 'dividas_amortizacao' in globals() else [],
    'dividas_custo': list(dividas_custo.keys()) if 'dividas_custo' in globals() else [],
    'dividas_inadimplencia': list(dividas_inadimplencia.keys()) if 'dividas_inadimplencia' in globals() else [],
    'dividas_atraso': list(dividas_atraso.keys()) if 'dividas_atraso' in globals() else []
}

# Medidas de escolaridade a avaliar
measures = [
    ('max', 'escolaridade_max'),
    ('median', 'escolaridade_median'),
    ('media', 'escolaridade_media')
]

# Tenta construir agregados de escolaridade a partir de `morador` se disponível
escola_df = None
if 'morador' in globals():
    try:
        if isinstance(morador, pl.DataFrame):
            cols = morador.columns
        else:
            cols = list(morador.columns)
        study_col = None
        for c in cols:
            cu = c.upper()
            if 'ANOS' in cu and 'ESTUDO' in cu:
                study_col = c
                break
        if study_col is not None:
            try:
                if isinstance(morador, pl.DataFrame):
                    escola_df = (morador.select([*key_cols, study_col])
                                 .group_by(key_cols)
                                 .agg([
                                     pl.col(study_col).max().alias('escolaridade_max'),
                                     pl.col(study_col).median().alias('escolaridade_median'),
                                     pl.col(study_col).mean().alias('escolaridade_media')
                                 ])
                                 .to_pandas())
                else:
                    escola_df = (morador.groupby(key_cols)[study_col]
                                 .agg(['max','median','mean'])
                                 .reset_index()
                                 .rename(columns={'max':'escolaridade_max','median':'escolaridade_median','mean':'escolaridade_media'}))
            except Exception:
                escola_df = None
    except Exception:
        escola_df = None

model_results = []

for grp_name, codes in debt_groups.items():
    # agrega despesas por UC para o grupo
    try:
        ind = despesa_ind.filter(pl.col('V9001').is_in(codes)).group_by(key_cols).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_ind'))
    except Exception:
        ind = pl.DataFrame({ 'COD_UPA': pl.Series([], dtype=pl.Utf8), 'NUM_DOM': pl.Series([], dtype=pl.Utf8), 'NUM_UC': pl.Series([], dtype=pl.Utf8), 'div_ind': pl.Series([], dtype=pl.Float64) })
    try:
        if 'despesa_col' in globals() and despesa_col.shape[0] > 0:
            col = despesa_col.filter(pl.col('V9001').is_in(codes)).group_by(key_cols).agg(pl.col('V8000').cast(pl.Float64).sum().alias('div_col'))
        else:
            col = pl.DataFrame({ 'COD_UPA': pl.Series([], dtype=pl.Utf8), 'NUM_DOM': pl.Series([], dtype=pl.Utf8), 'NUM_UC': pl.Series([], dtype=pl.Utf8), 'div_col': pl.Series([], dtype=pl.Float64) })
    except Exception:
        col = pl.DataFrame({ 'COD_UPA': pl.Series([], dtype=pl.Utf8), 'NUM_DOM': pl.Series([], dtype=pl.Utf8), 'NUM_UC': pl.Series([], dtype=pl.Utf8), 'div_col': pl.Series([], dtype=pl.Float64) })

    try:
        merged = morador_agg.join(ind, on=key_cols, how='left').join(col, on=key_cols, how='left')
        merged = merged.with_columns([
            pl.coalesce(pl.col('div_ind'), pl.lit(0)).alias('div_ind_f'),
            pl.coalesce(pl.col('div_col'), pl.lit(0)).alias('div_col_f')
        ])
        merged = merged.with_columns((pl.col('div_ind_f') + pl.col('div_col_f')).alias('total_div_group'))
        pdf = merged.to_pandas()
    except Exception as e:
        print('Erro ao construir agregados para', grp_name, ':', e)
        continue

    # Se escola_df existir, anexa colunas faltantes
    if escola_df is not None and isinstance(escola_df, pd.DataFrame):
        for mc in ['escolaridade_max','escolaridade_median','escolaridade_media']:
            if mc not in pdf.columns and mc in escola_df.columns:
                pdf = pdf.merge(escola_df[[*key_cols, mc]], on=key_cols, how='left')

    # detecta coluna de renda
    if 'renda_total' in pdf.columns:
        renda_col = 'renda_total'
    elif 'RENDA_TOTAL' in pdf.columns:
        renda_col = 'RENDA_TOTAL'
    else:
        renda_col = None

    if renda_col is not None:
        with np.errstate(divide='ignore', invalid='ignore'):
            pdf['razao_divida_renda'] = pdf['total_div_group'] / pdf[renda_col].replace({0:np.nan})
        pdf['log_renda'] = np.log(pdf[renda_col].fillna(0) + 1)
    else:
        pdf['razao_divida_renda'] = np.nan
        pdf['log_renda'] = np.nan

    # para cada medida de escolaridade, estima as duas amostras
    for measure_label, measure_col in measures:
        df_work = pdf.copy()
        if measure_col not in df_work.columns:
            df_work[measure_col] = np.nan
        df_work['escolaridade_used'] = df_work[measure_col]

        # filtro: obs com dependente e escolaridade
        df_model_all = df_work[(df_work['escolaridade_used'].notna()) & (df_work['razao_divida_renda'].notna())].copy()

        # AMOSTRA 1: todas as UCs
        samp_label = 'todas_UCs'
        n_all = len(df_model_all)
        if n_all >= 10:
            try:
                mod = smf.ols('razao_divida_renda ~ escolaridade_used + log_renda', data=df_model_all).fit(cov_type='HC3')
                est = mod.params.get('escolaridade_used', np.nan)
                se = mod.bse.get('escolaridade_used', np.nan)
                p = mod.pvalues.get('escolaridade_used', np.nan)
                r2 = mod.rsquared
            except Exception as e:
                print('Erro modelo (todas UCs) para', grp_name, measure_label, ':', e)
                est, se, p, r2 = [np.nan]*4
        else:
            est, se, p, r2 = [np.nan]*4

        model_results.append({
            'Grupo de Dívida': grp_name,
            'Medida': measure_label,
            'Amostra': samp_label,
            'N': n_all,
            'Coef_Escolaridade': est,
            'SE': se,
            't': (est / se) if (pd.notna(est) and pd.notna(se) and se!=0) else np.nan,
            'p_valor': p,
            'R2': r2
        })

        # AMOSTRA 2: apenas UCs com dívida>0
        samp_label = 'UCs_com_divida'
        df_pos = df_model_all[df_model_all['total_div_group'] > 0].copy()
        n_pos = len(df_pos)
        if n_pos >= 10:
            try:
                mod_pos = smf.ols('razao_divida_renda ~ escolaridade_used + log_renda', data=df_pos).fit(cov_type='HC3')
                est2 = mod_pos.params.get('escolaridade_used', np.nan)
                se2 = mod_pos.bse.get('escolaridade_used', np.nan)
                p2 = mod_pos.pvalues.get('escolaridade_used', np.nan)
                r22 = mod_pos.rsquared
            except Exception as e:
                print('Erro modelo (UCs com dívida) para', grp_name, measure_label, ':', e)
                est2, se2, p2, r22 = [np.nan]*4
        else:
            est2, se2, p2, r22 = [np.nan]*4

        model_results.append({
            'Grupo de Dívida': grp_name,
            'Medida': measure_label,
            'Amostra': samp_label,
            'N': n_pos,
            'Coef_Escolaridade': est2,
            'SE': se2,
            't': (est2 / se2) if (pd.notna(est2) and pd.notna(se2) and se2!=0) else np.nan,
            'p_valor': p2,
            'R2': r22
        })

# resultados consolidados
res_models_df = pd.DataFrame(model_results)

# formatação para exibição
display_df = res_models_df.copy()
display_df['Coef_Escolaridade'] = display_df['Coef_Escolaridade'].round(2)
display_df['SE'] = display_df['SE'].round(2)
display_df['t'] = display_df['t'].round(2)
display_df['p_valor'] = display_df['p_valor'].round(3)
display_df['R2'] = display_df['R2'].round(3)

# renomeia colunas para apresentação
display_df = display_df.rename(columns={
    'Grupo de Dívida': 'Grupo de Dívida',
    'Medida': 'Medida Escolaridade',
    'Amostra': 'Amostra',
    'N': 'Observações (N)',
    'Coef_Escolaridade': 'Coef. Escolaridade (anos)',
    'SE': 'Erro Padrão',
    't': 't',
    'p_valor': 'p-valor',
    'R2': 'R²'
})

# tabela Plotly final
col_names = list(display_df.columns)
col_vals = [display_df[c].astype(str).tolist() for c in col_names]
col_widths = [220, 160, 120, 160, 120, 100, 100, 100][:len(col_names)]

table_fig = go.Figure(data=[go.Table(
    header=dict(values=col_names, fill_color='lightgrey', align='left', font=dict(size=11)),
    cells=dict(values=col_vals, align='left', font=dict(size=10)),
    columnwidth=col_widths
)])

table_fig.update_layout(title='Modelos: escolaridade (max/median/média) vs razão dívida/renda — resultados', height=520, width=1200)

table_fig.show()

# gráfico de coeficientes (por medida/grupo/amostra)
if not res_models_df.empty:
    res_models_df['x_label'] = res_models_df['Grupo de Dívida'].astype(str) + ' | ' + res_models_df['Medida'].astype(str) + ' | ' + res_models_df['Amostra'].astype(str)
    coef_fig = go.Figure()
    coef_fig.add_trace(go.Bar(
        x=res_models_df['x_label'],
        y=res_models_df['Coef_Escolaridade'],
        error_y=dict(type='data', array=1.96 * res_models_df['SE'].fillna(0)),
        marker_color='#636EFA'
    ))
    coef_fig.update_layout(title='Coeficiente de escolaridade sobre razão dívida/renda (por grupo/medida/amostra)', yaxis_title='Coeficiente', xaxis_tickangle=45, height=600, width=1400)
    coef_fig.show()

# salva CSV final
out_fn = 'modelos_relacao_estudo_divida_medidas.csv'
res_models_df.to_csv(out_fn, index=False)
print(f'✓ Modelos estimados e salvos: {out_fn}')
